# Qiskit Fall Fest 2026 at KU
## Workshop 3. 양자 회로를 만들고 실행하기

앞선 강의에서는 상태와 게이트를 수식과 그림으로 살펴보았습니다. 이번에는 Qiskit으로 회로를 직접 작성하고, 게이트가 상태를 어떻게 바꾸는지 확인합니다. 이어서 같은 회로를 여러 번 측정하며 **측정 결과·shots·기댓값**의 의미를 익힙니다.

단일 큐비트에서 시작해 얽힘·CHSH·양자 텔레포테이션을 실행합니다. 이어서 VQA의 최적화 과정과 QEC의 오류 검출·보정을 직접 구현합니다. 모든 계산은 로컬 시뮬레이터에서 수행합니다. 기본 회로는 잡음 없이 실행하고, QEC 실습에서는 오류를 명시적으로 추가합니다.

| 순서 | 내용 |
|:--|:--|
| [실행 준비](#setup) | 패키지 설치와 노트북 사용법 |
| [Part 1](#gates) | 게이트, 상태벡터, 연산 순서 |
| [Part 2](#shots) | 측정 한 번, 여러 shots, 측정 후 상태 |
| [Part 3](#basis) | 측정 기저, 기댓값, Bloch 구면과 간섭 |
| [Part 4](#bell) | 두 큐비트의 표기와 Bell 상태 |
| [Part 5](#chsh) | CHSH 상관관계와 게임 |
| [Part 6](#teleportation) | 중간 측정과 양자 텔레포테이션 |
| [Part 7 · 선택](#primitives) | Sampler, Estimator, 매개변수 회로 |
| [Part 8](#vqa) | VQA: 에너지를 최소화하는 회로 찾기 |
| [Part 9](#qec) | QEC: 비트 반전 오류의 검출과 보정 |
| [해설](#solutions) | 연습문제 해설과 복습 |

**실습 방법** — 설명을 읽고 결과를 예상한 뒤, 바로 아래 코드 셀을 실행해 보세요. ‘직접 바꾸기’에서는 숫자나 게이트를 수정합니다. ‘연습’은 빈칸을 직접 채우는 문제이며, 해설은 뒤에 모았습니다. **선택**으로 표시한 셀과 연습문제를 건너뛰어도 다음 본문을 진행할 수 있습니다.

**교재와 함께 보기 · 2026-11-11 개정판**

Workshop 1은 기초, Workshop 2는 응용 이론, Workshop 3는 이 노트북의 실습입니다. 교재 4장은 Qiskit 문법, 5장은 Grover, 6장은 CHSH, 7장은 텔레포테이션, 8장은 QEC, 9장은 VQA입니다. 노트북은 VQA 다음에 QEC를 실행합니다. 영상 3강에 없는 Grover 코드는 교재 5장에서 선택 실습으로 다룹니다.

영상의 입력 C와 교재의 Q는 이 노트북의 q0와 같은 역할입니다. 이 노트북의 VQA는 H = −ZZ − XX이며, 교재의 H = Z + 0.4X는 별도의 확장 예제입니다.


### 처음 실습하시는 분께

이 강의는 Workshop1에서 **상태, 게이트, 측정, 중첩**을 배운 뒤 처음 Qiskit을 사용하는 분을 대상으로 합니다. Python까지 처음이라면 아래 문법 안내를 먼저 읽고, 영상을 멈추면서 셀을 한 개씩 실행하세요.

첫 번째 목표는 Part 1–4에서 **회로 만들기 → 실행 → 결과 읽기 → 한 가지 수정 → 재실행**을 스스로 해 보는 것입니다. Part 5–6의 CHSH와 텔레포테이션에서는 준비·측정·고전 처리를 구분하는 데 집중합니다. Part 7은 인터페이스 확장, Part 8–9의 VQA·QEC는 응용을 처음 경험하는 부분입니다. 처음부터 최적화 알고리즘이나 오류 정정 이론 전체를 숙달할 필요는 없습니다.

**따라 하는 순서:** 결과를 한 문장으로 예상하고 → 표시된 숫자나 게이트 한 가지를 바꾸고 → `Ctrl+Enter`로 현재 셀을 실행하고 → 새 출력이 예상과 같은지 확인합니다. `Shift+Enter`는 실행 후 다음 셀로 이동합니다. 코드를 고친 것만으로 이전 출력이 갱신되지는 않습니다. `[*]`가 보이면 계산 중이며, 완료된 뒤 결과를 읽습니다.

### Python 문법을 코드 한 줄씩 읽기

| 표기 | 이 노트북에서의 의미 |
|---|---|
| `qc = QuantumCircuit(1)` | 오른쪽에서 만든 회로를 `qc`라는 변수에 저장합니다. `=`는 대입이며 등식 검사가 아닙니다. |
| `qc.h(0)` | `qc` 회로의 `h` 메서드를 호출합니다. 괄호 안 `0`은 첫 번째 큐비트 번호입니다. |
| `sample_counts(qc, shots=128)` | 함수에 회로와 실행 횟수를 전달합니다. `shots=`는 인수 이름을 지정하는 방법입니다. |
| `counts.get("1", 0)` | 딕셔너리에서 결과 문자열 `"1"`의 횟수를 읽습니다. 해당 결과가 없으면 기본값 `0`을 받습니다. |
| `[16, 128, 1024]` | 여러 값을 순서대로 담은 리스트입니다. 첫 원소의 인덱스는 `0`입니다. |
| `for shots in shot_list:` | 리스트의 값을 하나씩 꺼내 아래 들여쓴 코드를 반복합니다. |
| `def f(x):` / `return y` | 함수를 정의하고, 호출한 곳으로 계산값을 돌려줍니다. 함수를 정의하는 것과 호출하는 것은 다릅니다. |
| `# 설명` | 주석입니다. `#` 뒤의 문자는 실행되지 않습니다. 게이트 줄을 주석 처리하면 그 게이트가 회로에서 빠집니다. |
| `True` / `False` | 참·거짓 값입니다. 예를 들어 `corrections=False`는 보정을 생략하도록 함수에 요청합니다. |
| `np.pi`, `1j` | 각각 원주율과 허수 단위입니다. 회전 각도는 라디안으로 입력합니다. |

여러 줄에 걸친 괄호 안에서는 줄을 나누어도 한 번의 함수 호출입니다. 반면 `def`, `for`, `if` 다음의 **들여쓰기**는 어느 코드가 그 블록에 속하는지를 결정합니다. 예제를 붙여 넣을 때도 들여쓰기를 유지하세요.


<a id="setup"></a>
## 실행 준비

JupyterLab 또는 VS Code에서 이 파일을 열고 Python 커널을 선택합니다. 셀을 클릭한 뒤 **Shift+Enter**를 누르면 실행하고 다음 셀로 이동합니다. 위에서부터 순서대로 실행하세요.

- 코드에서 `#` 뒤는 설명입니다. 각도는 라디안으로 입력합니다. 예를 들어 `np.pi / 2`는 $90^\circ$입니다.
- 처음 실행할 때 패키지가 없다면 아래 `%pip` 줄의 맨 앞 `#`를 지우고 실행한 뒤, **커널을 재시작**합니다. 이미 설치되어 있다면 그대로 넘어갑니다.
- 배포 파일에 저장된 그림은 이전 실행 결과입니다. 코드를 바꿨다면 해당 셀도 다시 실행해야 그림이 바뀝니다.
- 애니메이션이 보이지 않으면 해당 셀을 다시 실행하세요. 슬라이더에는 실행 중인 커널과 `ipywidgets` 지원이 필요합니다.

이 파일의 실행 확인 환경은 **Python 3.12 · Qiskit 2.2.3 · Qiskit Aer 0.17.2**입니다. 아래 버전은 재현을 위해 고정한 값입니다. IBM Quantum 계정이나 API 키는 필요하지 않습니다.

In [ ]:
# 처음 설치할 때만 아래 줄의 주석을 해제하세요. 설치 후 커널을 재시작하세요.
# %pip install "qiskit[visualization]==2.2.3" "qiskit-aer==0.17.2" "numpy<3" "scipy>=1.11,<2" "matplotlib>=3.8,<4" "ipywidgets>=8,<9" ipykernel

먼저 라이브러리와 시뮬레이터를 준비합니다. `QuantumCircuit`으로 회로를 만들고, `Statevector`로 상태를 계산하며, `AerSimulator`로 측정 결과를 얻습니다. `SEED`는 예제의 난수를 재현하기 위한 값입니다.

In [ ]:
import sys
import itertools
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import display, Markdown, HTML
import qiskit
import qiskit_aer
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter
from qiskit.quantum_info import (
    Statevector, DensityMatrix, Operator, Pauli, SparsePauliOp,
    partial_trace, state_fidelity,
)
from qiskit.visualization import plot_bloch_vector, array_to_latex
from qiskit_aer import AerSimulator

SEED = 2026
# 작은 교육용 회로에서 불필요한 병렬 처리 비용을 줄입니다.
simulator = AerSimulator(max_parallel_threads=2)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=["#3155C6", "#E58A32", "#17958D", "#875AB5"]),
    "animation.embed_limit": 32,
})
print("Python:", sys.version.split()[0])
print("Qiskit:", qiskit.__version__, "| Qiskit Aer:", qiskit_aer.__version__)

아래 셀은 회로·Bloch 구면·그래프를 그리는 보조 함수입니다. **한 번 실행한 뒤 Part 1로 넘어가세요.** 뒤에서 사용할 `sample_counts`는 회로를 시뮬레이터에 맞게 변환하고 실행해 측정 횟수를 돌려줍니다. Part 2에서는 이 실행 과정을 직접 나누어 살펴봅니다.

그래프의 축과 범례는 환경에 따른 한글 폰트 차이를 피하기 위해 영어로 표시합니다.

### 보조 함수는 어떻게 읽을까요?

`show_circuit(qc)`는 회로를 그리는 수업용 함수이고, `sample_counts(qc, shots=...)`는 `transpile → simulator.run → result → get_counts`를 묶은 수업용 함수입니다. Qiskit 자체의 함수 이름과 구분해 두세요. 먼저 이 정의 셀을 실행해야 아래에서 호출할 수 있습니다. 처음에는 내부의 모든 줄보다 **입력과 반환값**을 확인하면 됩니다. 아래쪽에서 실행 절차를 풀어 쓴 예제도 다시 다룹니다.

In [ ]:
def show_circuit(qc):
    fig = qc.draw(output="mpl", fold=28, idle_wires=True)
    display(fig)
    plt.close(fig)


def sample_counts(qc, shots=1024, seed=SEED):
    # 측정이 들어 있는 회로를 받습니다. 원본 회로는 바꾸지 않습니다.
    compiled = transpile(qc, simulator, optimization_level=0, seed_transpiler=SEED)
    return simulator.run(compiled, shots=shots, seed_simulator=seed).result().get_counts()


def bloch(state):
    return np.array([float(np.real(state.expectation_value(Pauli(p)))) for p in "XYZ"])


def show_bloch(states, titles):
    fig = plt.figure(figsize=(4 * len(states), 4.1))
    fig.subplots_adjust(top=0.82, bottom=0.04, wspace=0.08)
    for i, (state, title) in enumerate(zip(states, titles), 1):
        ax = fig.add_subplot(1, len(states), i, projection="3d")
        vector = bloch(state)
        plot_bloch_vector(vector, ax=ax)
        box = ax.get_position()
        fig.text((box.x0 + box.x1) / 2, 0.94, title, ha="center", va="center", fontsize=12)
        ax.view_init(elev=25, azim=35)
        if np.linalg.norm(vector) < 1e-8:
            ax.scatter([0], [0], [0], color="#3155C6", s=32)
    plt.show()


def compare_distribution(exact, counts, labels, title):
    total = sum(counts.values())
    x = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(6.6, 3.3))
    ax.bar(x - 0.18, [exact.get(k, 0) for k in labels], 0.36, label="Exact probability")
    ax.bar(x + 0.18, [counts.get(k, 0) / total for k in labels], 0.36, label=f"Frequency ({total} shots)")
    ax.set(xticks=x, xticklabels=labels, ylim=(0, 1.12), ylabel="Probability / frequency", title=title)
    ax.legend(fontsize=9)
    plt.show()


def bit_at(bitstring, index):
    # 공백은 고전 레지스터 사이의 구분입니다. 이 파일에서는 오른쪽부터 c0, c1, ...입니다.
    return int(bitstring.replace(" ", "")[-1 - index])


def marginal_counts(counts, index):
    return {str(v): sum(n for b, n in counts.items() if bit_at(b, index) == v) for v in (0, 1)}


def table(headers, rows):
    def escaped(value):
        return str(value).replace("|", "\\|").replace("<", "&lt;").replace(">", "&gt;")
    text = "| " + " | ".join(map(escaped, headers)) + " |\n"
    text += "| " + " | ".join(["---"] * len(headers)) + " |\n"
    text += "\n".join("| " + " | ".join(map(escaped, row)) + " |" for row in rows)
    display(Markdown(text))

<a id="gates"></a>
## Part 1. 게이트를 회로에 넣어 보기
### 1.1 첫 번째 회로: $X|0\rangle=|1\rangle$

`QuantumCircuit(1)`은 큐비트 하나를 갖는 회로를 만듭니다. `qc.x(0)`의 `0`은 **0번 큐비트**를 뜻합니다. 회로를 만드는 코드는 연산을 순서대로 기록하며, 아직 측정 결과를 얻는 단계는 아닙니다.

다음 회로는 초기 상태 $|0\rangle$에 $X$ 게이트를 적용합니다.

In [ ]:
qc_first = QuantumCircuit(1)
qc_first.x(0)
show_circuit(qc_first)

`Statevector.from_instruction`으로 회로의 연산을 계산해 봅니다. 별도의 상태 준비가 없으면 모든 큐비트는 $|0\rangle$에서 시작합니다.

$$X\begin{pmatrix}1\\0\end{pmatrix}=\begin{pmatrix}0\\1\end{pmatrix}.$$

In [ ]:
psi_first = Statevector.from_instruction(qc_first)
display(psi_first.draw("latex"))
print("상태벡터 [alpha, beta] =", psi_first.data)
print("Z 측정 확률 [P(0), P(1)] =", psi_first.probabilities())

상태 $|\psi\rangle=\alpha|0\rangle+\beta|1\rangle$에서 $\alpha,\beta$는 **확률진폭**입니다. Z 기저로 측정할 때의 확률은 $P(0)=|\alpha|^2$, $P(1)=|\beta|^2$입니다. 상태벡터는 시뮬레이터 내부의 계산 결과이며, 실제 장치에서 한 번 측정해 이 두 진폭을 모두 읽을 수 있는 것은 아닙니다.

### 1.2 $H$로 중첩 만들기

이번에는 $H$를 적용합니다. 두 확률진폭과 두 확률이 각각 얼마인지 비교하세요.

In [ ]:
qc_h = QuantumCircuit(1)
qc_h.h(0)
sv_h = Statevector.from_instruction(qc_h)
show_circuit(qc_h)
display(sv_h.draw("latex"))
print("확률진폭:", sv_h.data)
print("측정 확률:", sv_h.probabilities())
display(array_to_latex(Operator(qc_h).data, prefix="H = "))

두 진폭은 $1/\sqrt2\simeq0.7071$이지만, 두 확률은 각각 $1/2$입니다. `Operator(qc_h).data`는 회로 전체의 행렬을 보여 줍니다.

앞에서 배운 게이트를 Qiskit으로 적으면 다음과 같습니다.

| 게이트 | Qiskit 코드 | 작용 |
|:--|:--|:--|
| $X$ | `qc.x(0)` | $\vert 0\rangle$와 $\vert 1\rangle$ 교환 |
| $Y$ | `qc.y(0)` | $\vert 0\rangle\mapsto i\vert 1\rangle$, $\vert 1\rangle\mapsto-i\vert 0\rangle$ |
| $Z$ | `qc.z(0)` | $\vert 1\rangle$ 성분의 부호 반전 |
| $H$ | `qc.h(0)` | $\vert 0\rangle\leftrightarrow\vert +\rangle$, $\vert 1\rangle\leftrightarrow\vert -\rangle$ |
| $S,T$ | `qc.s(0)`, `qc.t(0)` | $\vert 1\rangle$ 성분에 $i,e^{i\pi/4}$ 곱하기 |
| $P(\phi)$ | `qc.p(phi, 0)` | $\vert 1\rangle$ 성분에 $e^{i\phi}$ 곱하기 |
| $R_x,R_y,R_z$ | `qc.rx(theta, 0)` 등 | Bloch 구면에서 각 축을 중심으로 회전 |

**직접 바꾸기:** 아래 `qc_gate.s(0)`을 `y`, `z`, `t`로 바꾸어 실행하세요. 먼저 $H$로 중첩을 만들었기 때문에 두 성분 사이의 상대위상 변화를 볼 수 있습니다. Python에서는 허수 단위를 `1j`로 씁니다.

In [ ]:
qc_gate = QuantumCircuit(1)
qc_gate.h(0)
qc_gate.s(0)                        # y(0), z(0), t(0)로 바꿔 보세요.
sv_gate = Statevector.from_instruction(qc_gate)
show_circuit(qc_gate)
display(sv_gate.draw("latex"))
print("상태벡터:", sv_gate.data)
print("Z 측정 확률:", sv_gate.probabilities())

### 1.3 게이트의 순서가 바뀌면

회로의 시간은 왼쪽에서 오른쪽으로 흐릅니다. 코드에서 $A$를 먼저, $B$를 나중에 쓰면 최종 상태는 $BA|\psi\rangle$입니다. 아래 두 회로는 측정 확률이 같아도 상태가 다를 수 있음을 보여줍니다.

$$|0\rangle\xrightarrow{H}|+\rangle\xrightarrow{Z}|-\rangle,
\qquad |0\rangle\xrightarrow{Z}|0\rangle\xrightarrow{H}|+\rangle.$$

In [ ]:
qc_hz = QuantumCircuit(1)
qc_hz.h(0)
qc_hz.z(0)
qc_zh = QuantumCircuit(1)
qc_zh.z(0)
qc_zh.h(0)
sv_hz = Statevector.from_instruction(qc_hz)
sv_zh = Statevector.from_instruction(qc_zh)
show_circuit(qc_hz)
show_circuit(qc_zh)
display(sv_hz.draw("latex"), sv_zh.draw("latex"))
print("H then Z:", sv_hz.probabilities())
print("Z then H:", sv_zh.probabilities())
print("전역위상을 제외하고 같은 상태인가요?", sv_hz.equiv(sv_zh))

두 회로는 각각 $|-\rangle$와 $|+\rangle$를 만듭니다. 상대적인 부호가 다르지만 Z 측정 확률은 같습니다. **Z 측정 확률만으로 상태 전체를 알 수는 없습니다.** Part 3에서 측정 기저를 바꾸어 두 상태를 구별합니다.

`equiv`는 전체에 곱해지는 **전역위상**을 제외하고 같은 상태인지 확인합니다. 예를 들어 $|\psi\rangle$와 $-|\psi\rangle$는 같은 물리적 상태지만, $|+\rangle$와 $|-\rangle$는 그렇지 않습니다.

각 예제는 새 회로를 만드는 줄부터 시작하므로, 같은 셀을 다시 실행해도 게이트가 계속 누적되지 않습니다.

### 연습 1. $|+i\rangle$ 만들기

초기 상태 $|0\rangle$에서 $H$와 $S$를 사용해 $|+i\rangle=(|0\rangle+i|1\rangle)/\sqrt2$를 만들어 보세요. 빈 줄에 두 게이트를 추가합니다. `goal_state`는 검사할 목표 상태입니다. [해설](#solution-1)

In [ ]:
exercise_1 = QuantumCircuit(1)
# 여기에 두 게이트를 작성하세요.


exercise_1_state = Statevector.from_instruction(exercise_1)
goal_state = Statevector([1, 1j] / np.sqrt(2))
show_circuit(exercise_1)
display(exercise_1_state.draw("latex"))
print("목표 상태가 만들어졌습니다." if exercise_1_state.equiv(goal_state)
      else "아직 목표 상태가 아닙니다. 게이트를 작성하거나 순서를 확인해 보세요.")

<a id="shots"></a>
## Part 2. 측정 한 번과 shots 여러 번

$|+\rangle$의 상태벡터는 두 확률진폭을 모두 포함합니다. 그러나 한 번의 Z 측정이 반환하는 값은 **0 또는 1 하나**입니다.

**1 shot**은 준비된 초기 상태에서 회로의 명령을 끝까지 수행하는 한 번의 시행입니다. **$N$ shots**는 같은 준비·연산·측정 절차를 $N$번 반복하는 것입니다. 한 큐비트를 준비한 뒤 재준비 없이 $N$번 연속 측정하는 절차와는 다릅니다. 한 shot 안에 여러 측정 명령이 들어갈 수도 있습니다.

먼저 상태를 만드는 회로와 측정 명령을 포함한 회로를 구분하겠습니다.

In [ ]:
qc_plus = QuantumCircuit(1)
qc_plus.h(0)
sv_plus = Statevector.from_instruction(qc_plus)

qc_plus_measured = QuantumCircuit(1, 1)   # 큐비트 1개, 결과를 기록할 고전 비트 1개
qc_plus_measured.h(0)
qc_plus_measured.measure(0, 0)            # q0의 결과를 c0에 기록
show_circuit(qc_plus_measured)

### 2.1 회로 실행과 결과 확인

`transpile`은 회로를 시뮬레이터가 실행할 수 있는 형태로 변환합니다. `run`에 `shots=16`을 주면 준비부터 측정까지 16번 반복합니다. `memory=True`는 각 shot의 결과도 저장하라는 뜻입니다.

In [ ]:
compiled_plus = transpile(qc_plus_measured, simulator, optimization_level=0,
                          seed_transpiler=SEED)
job = simulator.run(compiled_plus, shots=16, memory=True, seed_simulator=SEED)
result = job.result()
counts_16 = result.get_counts()
print("16 shots의 결과 순서:", result.get_memory())
print("counts:", counts_16)
print("counts의 합:", sum(counts_16.values()))

### 직접 바꾸기 · shots와 반환값

다음은 같은 실행을 보조 함수로 짧게 쓴 코드입니다. `shots`를 **16 → 128 → 1024 → 8192**로 바꾸고 매번 실행하세요. 출력의 합은 항상 shots와 같아야 합니다. `P(1)`은 이 실행에서 얻은 관측 비율이며 정확한 확률과 구분합니다. 같은 준비에서 shots만 바꾸는 실험입니다.

이 실험과 뒤의 shots 비교 반복문은 난수 시드가 다릅니다. 반복문은 `seed=SEED+i`를 사용하므로 횟수가 조금 달라도 정상입니다. 비교할 것은 횟수의 일치가 아니라 비율의 통계적 변동입니다.


In [ ]:
shots = 16
counts = sample_counts(qc_plus_measured, shots=shots)
print("counts:", counts)
print("total:", sum(counts.values()))
print("P(1):", counts.get("1", 0) / shots)

`counts`는 결과별 횟수를 모은 딕셔너리입니다. 예를 들어 `{'0': 7, '1': 9}`라면 0이 7회, 1이 9회 나왔다는 뜻입니다. 횟수를 전체 shots로 나누면 **관측 빈도**가 됩니다.

이론적인 확률은 각각 $1/2$이지만, 16번의 결과가 반드시 8회씩 나뉘지는 않습니다. 아래에서 정확한 확률과 관측 빈도를 비교합니다. `seed_simulator`는 난수 설정이며 shots나 잡음의 크기를 뜻하지 않습니다.

In [ ]:
print("정확한 확률:", sv_plus.probabilities_dict())
compare_distribution(sv_plus.probabilities_dict(), counts_16,
                     ["0", "1"], "Same preparation, 16 separate shots")

### 2.2 shots를 늘리면?

16, 128, 1024, 8192 shots를 비교해 봅니다. 회로는 같고 반복 횟수만 달라집니다.

**반복문 읽기:** `shot_list`는 비교할 실행 횟수 목록입니다. `zip`은 그래프 축과 shots를 하나씩 짝짓고, `enumerate`는 순서 번호 `i`도 붙입니다. 들여쓴 블록이 각 shots마다 한 번씩 실행됩니다. 먼저 아래 비교 실습처럼 숫자 하나만 바꿔 보고, 같은 작업을 반복문으로 묶는다고 생각하세요.

In [ ]:
shot_list = [16, 128, 1024, 8192]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.0), sharey=True)
shot_rows = []
for i, (ax, shots) in enumerate(zip(axes, shot_list)):
    counts = sample_counts(qc_plus_measured, shots=shots, seed=SEED+i)
    p1 = counts.get("1", 0) / shots
    ax.bar(["0", "1"], [counts.get("0", 0)/shots, p1])
    ax.axhline(0.5, color="#E58A32", ls="--", label="Exact = 0.5")
    ax.set(title=f"{shots} shots", ylim=(0, 0.85))
    shot_rows.append([shots, counts.get("1", 0), f"{p1:.4f}", f"{abs(p1-0.5):.4f}", f"{0.5/np.sqrt(shots):.4f}"])
axes[0].set_ylabel("Observed frequency")
plt.tight_layout()
plt.show()
table(["shots N", "n1", "p_hat(1)", "|p_hat - 0.5|", "theoretical SD"], shot_rows)

shots를 늘리면 관측 빈도가 이론적 확률 근처에 모이는 경향이 강해집니다. 다만 **한 번 얻은 오차가 항상 순서대로 줄어드는 것은 아닙니다.**

측정 결과 1의 확률을 $p$, 횟수를 $n_1$, 전체 shots를 $N$이라고 하면 $\hat p=n_1/N$입니다. 독립적인 시행에서 이 빈도의 표준편차는

$$\sigma_{\hat p}=\sqrt{\frac{p(1-p)}{N}}$$

입니다. $p=1/2$이면 $1/(2\sqrt N)$이므로, 전형적인 통계적 오차를 절반으로 줄이려면 shots를 약 4배 늘려야 합니다. shots를 늘려도 회로 자체에 넣은 잡음이나 장치의 체계적인 오류까지 없어지는 것은 아닙니다.

**선택 탐색:** 다음 셀은 하나의 실행 기록에서 누적 빈도를 그립니다. 연한 영역은 각 $N$에서 이론적 표준편차의 2배 범위이며, 전체 곡선을 동시에 포함하는 신뢰대역은 아닙니다.

In [ ]:
# 한 회로를 4096 shots 실행한 기록에서 누적 빈도를 계산합니다.
record = simulator.run(compiled_plus, shots=4096, memory=True, seed_simulator=SEED+20).result().get_memory()
outcomes = np.array([int(bit) for bit in record])
n = np.arange(1, len(outcomes)+1)
p_hat = np.cumsum(outcomes) / n
fig, ax = plt.subplots(figsize=(8, 3.3))
ax.plot(n, p_hat, lw=1, label="Running frequency")
ax.axhline(0.5, color="#E58A32", ls="--", label="Exact p = 0.5")
ax.fill_between(n, np.maximum(0, .5-1/np.sqrt(n)), np.minimum(1, .5+1/np.sqrt(n)),
                color="#3155C6", alpha=.12, label="Pointwise +/- 2 theoretical SD")
ax.set(xscale="log", xlabel="Number of shots", ylabel="Frequency of 1", ylim=(0,1))
ax.legend(fontsize=9)
plt.show()

### 2.3 같은 큐비트를 연속으로 측정하면?

이상적인 Z 측정에서 결과가 0이면 측정 후 상태는 $|0\rangle$, 결과가 1이면 $|1\rangle$입니다. 중간에 다른 연산 없이 다시 Z 측정하면 같은 결과가 나옵니다.

`Statevector.measure()`는 한 번의 결과와 그 결과에 해당하는 측정 후 상태를 반환합니다. 여기서 상태를 보여 주는 것은 시뮬레이터의 기능입니다.

In [ ]:
sv_before = Statevector.from_instruction(qc_plus)
sv_before.seed(SEED)
outcome, sv_after = sv_before.measure()
sv_after.seed(SEED+1)
outcome_again, _ = sv_after.measure()
print("첫 측정:", outcome, "| 재준비 없이 같은 Z 측정을 한 번 더:", outcome_again)
display(sv_after.draw("latex"))

이번에는 한 shot 안에 두 측정을 넣고, 이 회로 전체를 1024 shots 반복합니다. 두 측정 결과를 각각 `c0`, `c1`에 저장합니다. 출력 문자열은 왼쪽부터 `c1 c0` 순서입니다.

In [ ]:
# 같은 shot 안에서 연속으로 두 번 측정합니다. 중간 게이트나 reset이 없습니다.
qc_twice = QuantumCircuit(1, 2)
qc_twice.h(0)
qc_twice.measure(0, 0)
qc_twice.measure(0, 1)
show_circuit(qc_twice)
counts_twice = sample_counts(qc_twice, shots=1024)
print("counts (c1 c0):", counts_twice)
print("첫째와 둘째 결과가 다른 shot 수:", sum(n for b,n in counts_twice.items() if b[0] != b[1]))

`00`, `11`만 나타나며, 한 shot 안의 두 결과가 항상 같습니다. 반면 서로 다른 shots 사이에서는 상태를 다시 준비하므로 0과 1이 섞여 나옵니다.

**확인하기:** ‘한 상태를 준비한 뒤 Z 측정 두 번’과 ‘상태 준비부터 시작하는 2 shots’의 차이를 설명해 보세요.

<a id="basis"></a>
## Part 3. 측정 기저, 기댓값, Bloch 구면
### 3.1 측정 기저 바꾸기

앞에서 $|+\rangle$와 $|-\rangle$의 Z 측정 확률이 같다는 것을 확인했습니다. 두 상태를 구별하려면 $\{|+\rangle,|-\rangle\}$를 기준으로 하는 **X 기저 측정**이 필요합니다.

Qiskit의 `measure`는 Z 기저 측정입니다. 하지만 측정 직전에 $H$를 넣으면

$$H|+\rangle=|0\rangle,\qquad H|-\rangle=|1\rangle$$

이므로, 원래 상태를 X 기저로 측정한 것과 같은 결과를 얻습니다. 측정 전에 넣는 게이트는 **측정하려는 방향을 Z 방향으로 맞추는 역할**을 합니다.

In [ ]:
qc_x_measurement = QuantumCircuit(1, 1)
qc_x_measurement.h(0)
qc_x_measurement.z(0)             # 여기까지 |-> 준비
qc_x_measurement.barrier()
qc_x_measurement.h(0)             # X 기저를 Z 기저로 변환
qc_x_measurement.measure(0, 0)
show_circuit(qc_x_measurement)
print(sample_counts(qc_x_measurement, shots=1024))

**직접 바꾸기:** barrier 뒤의 `qc_x_measurement.h(0)`만 주석 처리하고 실행하면 같은 마이너스 상태를 Z 기저로 측정합니다. 0과 1이 비슷하게 나오는지 보세요. 그다음 주석을 해제하고 다시 실행하면 X 기저에서 모두 1이 나옵니다. 앞의 상태 준비 H·Z는 유지합니다. 다음 단계 전에 기저 변환 H를 복원하세요.

### 3.2 기댓값은 무엇일까요?

Z 측정에서 기록한 비트 0과 1에 각각 관측값 **$+1$과 $-1$**을 대응시킵니다. Z 연산자의 행렬이 $\operatorname{diag}(1,-1)$이기 때문입니다. 이 관측값의 확률 가중 평균을 **기댓값**이라고 합니다.

$$\langle Z\rangle=(+1)P(0)+(-1)P(1)=P(0)-P(1).$$

예를 들어 0의 확률이 $3/4$, 1의 확률이 $1/4$이면 $\langle Z\rangle=1/2$입니다. 한 번의 관측값이 $1/2$이라는 뜻이 아니라, **여러 결과의 평균이 가까워지는 값**입니다. 비트 0과 1을 그대로 평균한 값 $P(1)$과 구분하세요.

실험에서는 확률 대신 횟수를 사용해 $\langle Z\rangle$를 추정합니다.

$$\widehat{\langle Z\rangle}=\frac{n_0-n_1}{N}.$$

In [ ]:
expectation_prep = QuantumCircuit(1)
expectation_prep.ry(np.pi / 3, 0)   # P(0)=3/4, P(1)=1/4
expectation_state = Statevector.from_instruction(expectation_prep)
expectation_measured = expectation_prep.copy()
expectation_measured.measure_all()
expectation_counts = sample_counts(expectation_measured, shots=2048)
n0 = expectation_counts.get("0", 0)
n1 = expectation_counts.get("1", 0)
z_estimate = (n0 - n1) / (n0 + n1)
z_exact = float(np.real(expectation_state.expectation_value(Pauli("Z"))))
print("counts:", expectation_counts)
print(f"counts로 추정한 <Z>: {z_estimate:.4f}")
print(f"상태벡터에서 계산한 <Z>: {z_exact:.4f}")

`Pauli("Z")`는 관측량 $Z$를 나타내며, `expectation_value`는 상태로부터 기댓값을 계산합니다. 이 계산에는 유한 shots의 표본 오차가 없습니다.

X와 Y 측정에서도 결과 비트를 같은 방식으로 $+1,-1$에 대응시키면 $\langle X\rangle$, $\langle Y\rangle$를 얻습니다.

| 측정할 관측량 | Z 측정 직전에 적용할 게이트 | 비트와 관측값 |
|:--|:--|:--|
| $Z$ | 없음 | 0 → $+1$, 1 → $-1$ |
| $X$ | $H$ | 0 → $+1$, 1 → $-1$ |
| $Y$ | $S^\dagger$ 다음 $H$ | 0 → $+1$, 1 → $-1$ |

아래 함수에 상태 준비 회로와 기저 이름을 넣으면 해당 측정 회로를 만듭니다. `compose`는 이미 만든 회로를 이어 붙이는 메서드입니다.

**함수와 조건문:** `def measure_in_basis(preparation, basis):`는 두 입력을 받는 함수의 정의입니다. 정의만 실행하면 회로가 측정되는 것은 아닙니다. 다음 셀의 함수 호출이 실제 회로를 만듭니다. `if`·`elif`는 기저 이름에 따라 넣을 게이트를 고르고, `return qc`는 완성한 회로를 호출한 곳으로 돌려줍니다. `compose(..., inplace=True)`는 준비 회로를 현재 회로에 붙여 수정합니다.

In [ ]:
def measure_in_basis(preparation, basis):
    circuit = QuantumCircuit(1, 1)
    circuit.compose(preparation, qubits=[0], inplace=True)
    if basis == "X":
        circuit.h(0)
    elif basis == "Y":
        circuit.sdg(0)
        circuit.h(0)
    elif basis != "Z":
        raise ValueError("basis는 X, Y, Z 중 하나여야 합니다.")
    circuit.measure(0, 0)
    return circuit

In [ ]:
plus_prep = QuantumCircuit(1)
plus_prep.h(0)
minus_prep = QuantumCircuit(1)
minus_prep.h(0)
minus_prep.z(0)
rows = []
for name, prep in [("|+>", plus_prep), ("|->", minus_prep)]:
    for basis in ["Z", "X"]:
        counts = sample_counts(measure_in_basis(prep, basis), shots=1024)
        expectation = (counts.get("0",0)-counts.get("1",0)) / 1024
        rows.append([name, basis, str(counts), f"{expectation:.3f}"])
table(["Input", "Basis", "counts", "Expectation estimate"], rows)
show_circuit(measure_in_basis(minus_prep, "X"))

Z 측정에서는 두 상태 모두 0과 1이 비슷하게 나옵니다. X 측정에서는 $|+\rangle$의 결과가 항상 0, $|-\rangle$의 결과가 항상 1이므로 각각 $\langle X\rangle=+1,-1$입니다.

**직접 바꾸기:** `minus_prep.z(0)`을 `minus_prep.s(0)`으로, `basis` 목록을 `["Z", "Y"]`로 바꾸어 $|+i\rangle$의 Y 측정 결과를 확인하세요.

<a id="bloch"></a>
### 3.3 Bloch 구면에서 상태 보기

단일 큐비트의 순수상태는 전역위상을 제외하고

$$|\psi\rangle=\cos\frac{\theta}{2}|0\rangle+
 e^{i\phi}\sin\frac{\theta}{2}|1\rangle$$

로 쓸 수 있습니다. 이를 나타내는 **Bloch 벡터**는

$$\mathbf r=(\langle X\rangle,\langle Y\rangle,\langle Z\rangle)
=(\sin\theta\cos\phi,\sin\theta\sin\phi,\cos\theta)$$

입니다. 그림의 화살표는 복소수 두 성분으로 된 상태벡터 자체가 아니라, 그 상태에 대응하는 실수 3차원 벡터입니다. $|0\rangle$은 북극, $|1\rangle$은 남극에 놓입니다.

In [ ]:
show_bloch(
    [Statevector.from_label("0"), Statevector.from_label("1"), Statevector.from_label("+")],
    [r"$|0\rangle$", r"$|1\rangle$", r"$|+\rangle$"],
)

세 축의 좌표가 각각 X·Y·Z 기댓값입니다. 순수상태는 구면 위에 놓이고, 북극 $|0\rangle$의 좌표는 $(0,0,1)$, $|+\rangle$의 좌표는 $(1,0,0)$입니다.

앞에서 다룬 측정도 그려 봅니다. 왼쪽은 측정 전 $|+\rangle$, 오른쪽은 한 번의 Z 측정 결과를 알고 있을 때의 상태입니다.

In [ ]:
bloch_before = Statevector.from_label("+")
bloch_before.seed(SEED)
bloch_outcome, bloch_after = bloch_before.measure()
show_bloch([bloch_before, bloch_after],
           ["Before Z measurement", f"After Z: result {bloch_outcome}"])

### 3.4 회전 게이트를 이어 붙이기

다음 회로는 $R_y(\pi/2)$, $R_z(\pi/2)$, $R_x(\pi/2)$ 순서로 작용합니다. 중간 상태도 따로 계산해 보겠습니다.

$$R_k(\gamma)=e^{-i\gamma\sigma_k/2}.$$

아래 애니메이션은 각 게이트의 각도를 0부터 지정값까지 변화시키며 `Statevector`를 다시 계산합니다. 회전의 수학적 경로이며, 실제 장치의 펄스 파형이나 실행 시간을 표시한 것은 아닙니다.

In [ ]:
qc_path = QuantumCircuit(1)
path_states = [Statevector.from_instruction(qc_path)]
qc_path.ry(np.pi / 2, 0)
path_states.append(Statevector.from_instruction(qc_path))
qc_path.rz(np.pi / 2, 0)
path_states.append(Statevector.from_instruction(qc_path))
qc_path.rx(np.pi / 2, 0)
path_states.append(Statevector.from_instruction(qc_path))
show_circuit(qc_path)
show_bloch(path_states, ["Initial", "After RY(pi/2)", "After RZ(pi/2)", "After RX(pi/2)"])

# 다음 셀에서 위 회로의 회전 경로를 애니메이션으로 그립니다.
rotation_sequence = [("ry", np.pi/2), ("rz", np.pi/2), ("rx", np.pi/2)]

In [ ]:
# 화살표의 매 프레임을 Qiskit으로 계산합니다. 외부 동영상 파일은 만들지 않습니다.
trajectory, frame_labels = [], []
completed = QuantumCircuit(1)
for gate, final_angle in rotation_sequence:
    for angle in np.linspace(0, final_angle, 25):
        frame_qc = completed.copy()
        getattr(frame_qc, gate)(float(angle), 0)
        trajectory.append(bloch(Statevector.from_instruction(frame_qc)))
        frame_labels.append(f"{gate.upper()}({angle / np.pi:.2f} pi)")
    getattr(completed, gate)(final_angle, 0)
trajectory = np.array(trajectory)
# Qiskit의 Bloch 그림 좌표는 (y, -x, z) 순서로 배치됩니다.
plot_trajectory = trajectory[:, [1, 0, 2]] * np.array([1, -1, 1])

fig = plt.figure(figsize=(6.0, 4.5))
fig.subplots_adjust(top=0.80, bottom=0.05)
ax = fig.add_subplot(111, projection="3d")
plot_bloch_vector([0, 0, 0], ax=ax)
ax.view_init(elev=25, azim=35)
line, = ax.plot([], [], [], color="#3155C6", linewidth=3)
point, = ax.plot([], [], [], "o", color="#3155C6", markersize=7)
trail, = ax.plot([], [], [], color="#E58A32", linewidth=2, alpha=0.8)

def update_bloch_frame(frame):
    vector = plot_trajectory[frame]
    line.set_data_3d([0, vector[0]], [0, vector[1]], [0, vector[2]])
    point.set_data_3d([vector[0]], [vector[1]], [vector[2]])
    trail.set_data_3d(plot_trajectory[:frame + 1, 0], plot_trajectory[:frame + 1, 1], plot_trajectory[:frame + 1, 2])
    ax.set_title(frame_labels[frame] + f"   |   P(1)={(1-vector[2])/2:.3f}", pad=32)
    return line, point, trail

bloch_animation = FuncAnimation(fig, update_bloch_frame, frames=len(trajectory), interval=90, blit=False)
bloch_animation_html = bloch_animation.to_jshtml(fps=12, default_mode="once")
# 재생 버튼도 외부 아이콘 폰트 없이 표시합니다.
import re
bloch_animation_html = re.sub(r"<link[^>]*>", "", bloch_animation_html)
icon_labels = {
    "fa-minus": "−", "fa-plus": "+", "fa-fast-backward": "|◀",
    "fa-step-backward": "◀|", "fa-play fa-flip-horizontal": "◀",
    "fa-pause": "Ⅱ", "fa-play": "▶", "fa-step-forward": "|▶",
    "fa-fast-forward": "▶|",
}
for icon_class, label in icon_labels.items():
    bloch_animation_html = bloch_animation_html.replace(
        '<i class="fa ' + icon_class + '"></i>', label)

plt.close(fig)
display(HTML(bloch_animation_html))

**재생 버튼을 눌러 확인하세요.** $R_z$가 작용하는 동안 화살표의 높이, 즉 $\langle Z\rangle$와 $P(1)$은 변하지 않습니다. 상태의 상대위상은 변하지만 Z 측정 확률은 그대로인 예입니다.

### 3.5 직접 각도 바꾸기 · 선택

`theta`와 `phi`로 입력 상태를 만들고, `gate`와 `angle`로 추가 회전을 지정합니다. 아래 함수의 핵심은 `ry → rz → 선택한 회전` 세 줄입니다. $R_z(\phi)R_y(\theta)|0\rangle$은 위에서 적은 상태와 전역위상만 다릅니다.

In [ ]:
def rotation_lab(theta=np.pi/2, phi=0.0, gate="rz", angle=np.pi/2):
    circuit = QuantumCircuit(1)
    circuit.ry(theta, 0)
    circuit.rz(phi, 0)
    before = Statevector.from_instruction(circuit)
    getattr(circuit, gate)(angle, 0)
    after = Statevector.from_instruction(circuit)
    show_bloch([before, after], ["Input", f"After {gate.upper()}({angle/np.pi:.2f} pi)"])
    print("Input Bloch vector:", bloch(before))
    print("Output Bloch vector:", bloch(after))
    print("Output [P(0), P(1)]:", after.probabilities())

# 슬라이더를 지원하지 않는 환경에서도 이 줄의 숫자를 바꾸어 실습할 수 있습니다.
rotation_lab(theta=np.pi/2, phi=0.0, gate="rz", angle=np.pi/2)

In [ ]:
# 선택: 실행 중인 커널이 있는 JupyterLab / VS Code에서 사용합니다.
try:
    import ipywidgets as widgets
    controls = widgets.interactive(
        rotation_lab,
        theta=widgets.FloatSlider(value=np.pi/2, min=0, max=np.pi, step=np.pi/24, description="theta", continuous_update=False),
        phi=widgets.FloatSlider(value=0, min=0, max=2*np.pi, step=np.pi/24, description="phi", continuous_update=False),
        gate=widgets.Dropdown(options=["rx", "ry", "rz"], value="rz", description="gate"),
        angle=widgets.FloatSlider(value=np.pi/2, min=-2*np.pi, max=2*np.pi, step=np.pi/24, description="angle", continuous_update=False),
    )
    display(controls)
except ImportError:
    print("ipywidgets가 없어도 위 셀의 rotation_lab(...) 인수를 바꾸어 실행할 수 있습니다.")

**확인하기:** $|0\rangle$에 $R_z$만 적용하면 Bloch 벡터가 움직일까요? $|+\rangle$에 $R_z(\pi)$를 적용하면 어느 방향을 향할까요?

슬라이더를 사용할 수 없다면 위 `rotation_lab(...)`의 숫자를 바꾸고 셀을 다시 실행하면 됩니다. 두 선택 셀을 모두 건너뛰어도 다음 실습에 영향이 없습니다.

<a id="interference"></a>
### 3.6 상대위상을 측정 결과로 바꾸기

이제 측정 기저와 위상 변화를 연결해 봅니다. 첫 $H$로 $|+\rangle$를 만들고, $P(\phi)$로 상대위상을 바꾼 뒤, 마지막 $H$와 Z 측정으로 X 기저의 정보를 읽습니다.

$$H P(\phi) H|0\rangle
=\frac{1+e^{i\phi}}{2}|0\rangle+\frac{1-e^{i\phi}}{2}|1\rangle.$$

마지막 $H$에서 두 성분의 진폭이 더해지거나 상쇄됩니다. 따라서

$$P(0)=\cos^2\frac{\phi}{2},\qquad P(1)=\sin^2\frac{\phi}{2}.$$

먼저 $\phi=\pi/3$인 한 회로를 실행합니다.

In [ ]:
phase = np.pi / 3
interferometer = QuantumCircuit(1)
interferometer.h(0)
interferometer.p(phase, 0)
interferometer.h(0)
show_circuit(interferometer)
print("정확한 [P(0), P(1)]:", Statevector.from_instruction(interferometer).probabilities())

이번에는 첫 $H$ 다음에 Z 측정을 삽입한 회로도 만듭니다. 중간 결과로 분류하지 않고 모든 시행을 모으면, 최종 결과는 위상에 관계없이 $P(0)=1/2$가 됩니다.

아래 함수에서 `detector=True`이면 중간 결과를 `c0`, 최종 결과를 `c1`에 저장합니다. 중간 측정이 없을 때는 최종 결과만 `c0`에 저장합니다.

In [ ]:
def measured_interferometer(phi, detector=False):
    circuit = QuantumCircuit(1, 2 if detector else 1)
    circuit.h(0)
    if detector:
        circuit.measure(0, 0)        # c0: 중간 측정 결과
    circuit.p(float(phi), 0)
    circuit.h(0)
    circuit.measure(0, 1 if detector else 0)  # 최종 측정 결과
    return circuit


show_circuit(measured_interferometer(np.pi/3, detector=True))

위상을 0부터 $2\pi$까지 바꾸면서 두 회로를 각각 1024 shots 실행합니다. 실선은 중간 측정이 없을 때의 이론적 확률입니다.

In [ ]:
SHOTS_INTERFERENCE = 1024
phase_values = np.linspace(0, 2*np.pi, 25)
exact_fringe, sampled_fringe, measured_fringe = [], [], []
for i, phi in enumerate(phase_values):
    exact_fringe.append(np.cos(phi/2)**2)
    c = sample_counts(measured_interferometer(phi), shots=SHOTS_INTERFERENCE, seed=SEED+i)
    sampled_fringe.append(c.get("0",0)/SHOTS_INTERFERENCE)
    c_mid = sample_counts(measured_interferometer(phi, detector=True), shots=SHOTS_INTERFERENCE, seed=SEED+100+i)
    measured_fringe.append(marginal_counts(c_mid, 1)["0"]/SHOTS_INTERFERENCE)

fig, ax = plt.subplots(figsize=(8,3.6))
ax.plot(phase_values/np.pi, exact_fringe, label="Exact, no middle measurement")
ax.scatter(phase_values/np.pi, sampled_fringe, s=22, label=f"{SHOTS_INTERFERENCE} shots, no middle measurement")
ax.plot(phase_values/np.pi, measured_fringe, "o--", ms=4, label=f"{SHOTS_INTERFERENCE} shots, middle Z measurement")
ax.set(xlabel=r"Phase $\phi/\pi$", ylabel="Final P(0)", ylim=(-.05,1.05), xticks=[0,.5,1,1.5,2])
ax.legend(fontsize=9, loc="upper center")
plt.show()

중간 측정이 없으면 상대위상에 따라 간섭무늬가 나타납니다. 중간 Z 측정의 결과를 무시하고 모으면, 중첩의 두 성분 사이의 결맞음이 사라져 간섭무늬도 사라집니다. 사람이 결과를 읽었는지가 아니라 **회로에 측정 연산이 들어갔는지**가 차이를 만듭니다.

**직접 바꾸기:** 위 셀의 `SHOTS_INTERFERENCE = 1024`를 64로 바꾸어 보세요. 확률 곡선은 그대로지만 관측 빈도의 흔들림이 커집니다.

### 연습 2. 위상을 측정 결과로 바꾸기

$|0\rangle$에 $H$, $P(\pi)$, $H$를 차례대로 가한 회로를 작성하고, 최종 Z 측정 결과가 항상 1인지 확인하세요. `phase_ex2`를 $\pi/2$로 바꾸면 어떻게 될까요?

[해설](#solution-2)

In [ ]:
phase_ex2 = np.pi
exercise_2 = QuantumCircuit(1, 1)
# 여기에 H, P(phase_ex2), H를 작성하세요.


exercise_2.measure(0, 0)
counts_ex2 = sample_counts(exercise_2, shots=512)
show_circuit(exercise_2)
print("counts:", counts_ex2)

<a id="bell"></a>
## Part 4. 두 큐비트와 Bell 상태

### 4.1 먼저 비트 순서 확인하기

Qiskit 회로의 맨 위 선이 $q_0$이지만, 상태벡터의 기저 표기는 **$|q_1q_0\rangle$** 순서입니다. `measure([0,1],[0,1])`을 사용했을 때 counts의 문자열도 **`c1 c0`** 순서로 읽습니다.

| 회로 | Qiskit 상태 표기 | 측정 문자열 |
|:--|:--|:--|
| $q_0$에만 $X$ | $\vert 01\rangle$ | `01` |
| $q_1$에만 $X$ | $\vert 10\rangle$ | `10` |

아래 예제에서 오른쪽 비트가 어떤 큐비트에서 나온 값인지 확인하세요. 이후 상태벡터·Pauli 문자열·측정 결과도 이 규칙을 따릅니다.

In [ ]:
qc_order = QuantumCircuit(2)
qc_order.x(0)
show_circuit(qc_order)
display(Statevector.from_instruction(qc_order).draw("latex"))
qc_order_m = qc_order.copy()
qc_order_m.measure_all()
print(sample_counts(qc_order_m, shots=16))

### 4.2 $H$와 CNOT으로 얽힘 만들기

`cx(control, target)`은 제어 큐비트가 1일 때 표적 큐비트를 뒤집습니다. CNOT을 가했다고 언제나 얽힘이 생기지는 않습니다. 다음처럼 제어 큐비트에 먼저 중첩을 만들면 Bell 상태가 됩니다.

$$|00\rangle\xrightarrow{H_0}\frac{|00\rangle+|01\rangle}{\sqrt2}
\xrightarrow{\mathrm{CX}_{0\to1}}|\Phi^+\rangle
=\frac{|00\rangle+|11\rangle}{\sqrt2}.$$

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
sv_bell = Statevector.from_instruction(bell)
show_circuit(bell)
display(sv_bell.draw("latex"))

**직접 바꾸기:** `bell.cx(0, 1)`만 잠시 주석 처리하면 상태는 $(|00\rangle+|01\rangle)/\sqrt2$입니다. 다시 CX를 넣고 실행하면 $|01\rangle$ 성분이 $|11\rangle$로 바뀝니다. 이 비교를 마친 뒤에는 CX를 복원한 상태로 아래 셀을 실행하세요.

**왜 이 상태를 얽힘이라고 할까요?**

두 큐비트의 순수상태가 각 큐비트 상태의 곱으로 표현되면 곱상태라고 합니다. Qiskit의 $q_1q_0$ 순서에서

$$(a|0\rangle+b|1\rangle)_{q_1}\otimes(c|0\rangle+d|1\rangle)_{q_0}
=ac|00\rangle+ad|01\rangle+bc|10\rangle+bd|11\rangle.$$

여기서 $\otimes$는 두 계의 상태를 합쳐 전체 상태를 만드는 텐서곱입니다. 이 식이 Bell 상태라면 $ac=bd=1/\sqrt2$인 동시에 $ad=bc=0$이어야 합니다. 그런데 $ac$와 $bd$가 모두 0이 아니면 $a,b,c,d$도 모두 0이 아니므로, 나머지 두 곱을 0으로 만들 수 없습니다.

따라서 Bell 상태는 **각 큐비트의 순수상태 두 개로 분리해서 쓸 수 없는 얽힌 상태**입니다. 이 곱상태 판정은 순수상태에 대한 설명입니다. 혼합상태의 얽힘은 곱상태들의 확률적 혼합으로 표현할 수 있는지까지 따져야 합니다.

측정은 원본 회로의 복사본에 추가합니다. `measure_all()`은 모든 큐비트의 결과를 저장할 고전 비트를 만들고 측정 명령을 붙입니다.

In [ ]:
bell_m = bell.copy()
bell_m.measure_all()
counts_bell = sample_counts(bell_m, shots=2048)
compare_distribution(sv_bell.probabilities_dict(), counts_bell,
                     ["00","01","10","11"], "Bell state: Z-basis outcomes")
print("q0의 결과만 모으면:", marginal_counts(counts_bell, 0))
print("q1의 결과만 모으면:", marginal_counts(counts_bell, 1))

두 결과는 각각 무작위지만 Z 기저에서는 항상 같습니다. 다만 이 히스토그램만으로 얽힘을 확인할 수는 없습니다. $|00\rangle$와 $|11\rangle$를 반반 준비한 혼합상태도 똑같은 Z 분포를 줍니다.

### 4.3 중첩과 혼합, 그리고 한 큐비트만의 상태

**중첩**은 하나의 상태 안에 확률진폭과 상대위상이 함께 있는 것입니다. **혼합**은 여러 상태를 확률적으로 섞어 기술하는 것입니다. 두 경우를 함께 표현하는 방법이 밀도행렬 $\rho$입니다.

순수상태에는 $\rho=|\psi\rangle\langle\psi|$를, 확률 $p_k$로 상태 $|\psi_k\rangle$가 준비되는 혼합에는 $\rho=\sum_k p_k|\psi_k\rangle\langle\psi_k|$를 사용합니다. $|\psi\rangle\langle\psi|$는 상태 열벡터와 그 켤레전치 행벡터의 곱입니다.

예를 들어 $|+\rangle$와 $|0\rangle,|1\rangle$의 반반 혼합은

$$\rho_+=\frac12\begin{pmatrix}1&1\\1&1\end{pmatrix},\qquad
\rho_{\mathrm{mix}}=\frac12\begin{pmatrix}1&0\\0&1\end{pmatrix}=\frac I2.$$

두 상태의 Z 측정 확률은 같습니다. 하지만 $\rho_+$의 X 측정은 항상 $+1$을 주고, $I/2$의 X 측정은 $\pm1$을 반반 줍니다. 이 예에서는 비대각 원소가 상대위상의 효과가 남아 있는지를 보여 줍니다. Part 3에서 중간 Z 측정의 결과를 무시했을 때도 $\rho_+$가 $I/2$로 바뀌어 간섭무늬가 사라졌습니다.

Bell 상태의 각 큐비트만 보면 $I/2$입니다. 상대 큐비트를 직접 측정하거나 버리는 연산을 수행한다는 뜻이 아니라, **한쪽에서 얻을 수 있는 모든 측정 통계만 기술한 상태**입니다. 이 상태를 구하는 계산이 부분대각합이며, `partial_trace(state, [1])`은 q1을 제외하고 q0의 상태를 구합니다.

단일 큐비트의 $I/2$는 X·Y·Z 기댓값이 모두 0이므로 Bloch 구면의 원점에 놓입니다. 순수상태는 구면 위에, 혼합상태는 구 내부에 놓입니다. **각 큐비트가 혼합상태라는 사실과 전체 Bell 상태가 순수상태라는 사실은 양립합니다.**

In [ ]:
rho_q0 = partial_trace(sv_bell, [1])
rho_q1 = partial_trace(sv_bell, [0])
display(array_to_latex(rho_q0.data, prefix=r"\rho_{q_0} = "))
show_bloch([rho_q0, rho_q1], ["q0 reduced state: I/2", "q1 reduced state: I/2"])
print("Bloch(q0):", bloch(rho_q0))

두 큐비트의 **상관관계**는 두 관측값의 곱을 평균하여 구합니다. 예를 들어 두 큐비트 모두 X를 측정하면

$$\langle XX\rangle=P(00)+P(11)-P(01)-P(10)$$

입니다. 같은 비트끼리는 곱이 $+1$, 다른 비트끼리는 곱이 $-1$이기 때문입니다. Qiskit의 Pauli 문자열도 $q_1q_0$ 순서이므로 `"ZX"`는 q1의 Z와 q0의 X를 뜻합니다.

아래에서는 Bell 상태와 $|00\rangle,|11\rangle$의 반반 혼합을 비교합니다. `DensityMatrix(np.diag(...))`의 대각 원소는 이 경우 각 기저 상태의 혼합 비율입니다.

In [ ]:
classical_mix = DensityMatrix(np.diag([0.5, 0, 0, 0.5]))
table(["State", "<ZZ>", "<XX>", "<YY>"], [
    [name] + [f"{np.real(state.expectation_value(Pauli(p))):.1f}" for p in ["ZZ","XX","YY"]]
    for name,state in [("Bell", sv_bell), ("Classical mixture", classical_mix)]
])

표에서 Bell 상태는 $\langle XX\rangle=1$, $\langle YY\rangle=-1$입니다. 즉 $|\Phi^+\rangle$를 같은 Y 기저로 측정하면 결과는 **반대**입니다. “같은 방향으로 측정하면 언제나 같은 결과”라고 일반화하면 안 됩니다. 다음 절에서는 여러 측정 방향의 상관관계를 함께 확인합니다.

각 큐비트의 Bloch 벡터 두 개만으로는 이 상관관계까지 표현할 수 없습니다.

### 연습 3. 다른 Bell 상태 만들기

$|\Psi^-\rangle=(|01\rangle-|10\rangle)/\sqrt2$를 만들어 보세요. 먼저 $|\Phi^+\rangle$를 만든 뒤 한 큐비트에 $X,Z$를 적용하는 방법이 있습니다. 전역위상만 다르면 같은 물리적 상태로 판정합니다. [해설](#solution-3)

In [ ]:
exercise_3 = QuantumCircuit(2)
# Bell 상태를 만들고, 필요한 단일 큐비트 게이트를 추가하세요.


sv_ex3 = Statevector.from_instruction(exercise_3)
goal_bell = Statevector([0, 1, -1, 0] / np.sqrt(2))
show_circuit(exercise_3)
display(sv_ex3.draw("latex"))
print("목표 상태가 만들어졌습니다." if sv_ex3.equiv(goal_bell)
      else "아직 목표 상태가 아닙니다. 게이트와 순서를 확인해 보세요.")

<a id="chsh"></a>
## Part 5. CHSH: 네 가지 측정 설정으로 상관관계 비교하기

Alice는 $q_0$, Bob은 $q_1$을 갖습니다. 각각 입력 비트 $x,y\in\{0,1\}$에 따라 측정 방향을 고릅니다. 측정 결과 비트를 $a,b$라고 하면 관측값은 $(-1)^a,(-1)^b$입니다.

$$E_{xy}=\langle A_xB_y\rangle
=P(a=b\mid x,y)-P(a\ne b\mid x,y),$$
$$S=E_{00}+E_{01}+E_{10}-E_{11}.$$

국소 숨은변수 모형에서는 $|S|\le2$입니다. 측정 설정의 선택이 숨은변수와 독립이고, 각 측의 결과가 상대 측 설정에 의존하지 않는다는 조건이 필요합니다. 아래 계산은 양자모형의 예측을 **고전 컴퓨터로 시뮬레이션**하는 것이며, 장치를 이용한 Bell 실험 자체는 아닙니다.

### 5.1 측정 방향을 회로로 바꾸기

XZ 평면에서 $O(\gamma)=\cos\gamma Z+\sin\gamma X$를 측정하려면, **$R_y(-\gamma)$를 적용한 뒤 Z 측정**합니다.

$$R_y(-\gamma)^\dagger ZR_y(-\gamma)=\cos\gamma Z+\sin\gamma X.$$

| 입력 | 측정 관측량 | 각도 $\gamma$ | 측정 전 게이트 |
|:--|:--|:--|:--|
| Alice $x=0$ | $A_0=Z$ | $0$ | 없음 |
| Alice $x=1$ | $A_1=X$ | $\pi/2$ | $R_y(-\pi/2)$ |
| Bob $y=0$ | $B_0=(Z+X)/\sqrt2$ | $\pi/4$ | $R_y(-\pi/4)$ |
| Bob $y=1$ | $B_1=(Z-X)/\sqrt2$ | $-\pi/4$ | $R_y(+\pi/4)$ |

이 설정과 $|\Phi^+\rangle$에서 $E_{00}=E_{01}=E_{10}=1/\sqrt2$, $E_{11}=-1/\sqrt2$이므로 $S=2\sqrt2$입니다.

**고전적 한계는 왜 2일까요?**

국소적인 결정론 전략 하나를 생각해 봅시다. Alice는 설정별로 $A_0,A_1\in\{+1,-1\}$을, Bob은 $B_0,B_1\in\{+1,-1\}$을 출력하도록 정해 둡니다. 자신의 출력은 상대방이 선택한 설정에 의존하지 않습니다. 이 전략의 CHSH 조합은

$$s=A_0B_0+A_0B_1+A_1B_0-A_1B_1
=A_0(B_0+B_1)+A_1(B_0-B_1).$$

$B_0=B_1$이면 두 번째 괄호가 0이고 첫 번째가 $\pm2$입니다. $B_0=-B_1$이면 반대입니다. 어느 경우든 $s=\pm2$입니다. 공유 난수로 여러 전략을 섞어도 이 값들을 평균할 뿐이므로 $|S|\le2$입니다. 여기서는 측정 설정을 선택하는 난수가 공유된 숨은변수와 독립이라고 가정합니다.

**측정 방향을 왜 이렇게 골랐을까요?**

$|\Phi^+\rangle$를 XZ 평면의 각도 $\alpha,\beta$에서 측정하면 상관관계는 $E(\alpha,\beta)=\cos(\alpha-\beta)$입니다. Alice의 방향을 $0,\pi/2$, Bob의 방향을 $+\pi/4,-\pi/4$로 잡으면 앞의 세 상관관계는 $1/\sqrt2$, 마지막은 $-1/\sqrt2$가 됩니다. 따라서 $S=2\sqrt2$입니다. 이 측정 방향들은 부호가 다른 마지막 항까지 $S$를 크게 만드는 쪽으로 기여하게 합니다.

아래 시뮬레이션은 이 양자이론의 예측을 재현합니다. 시뮬레이션 수치만으로 실제 장치의 비국소성을 입증하는 것은 아닙니다.

**처음 읽는 순서:** `chsh_circuit`의 입력 `x`, `y`는 측정 설정이고 측정 결과가 아닙니다. `bob_angle`은 측정 방향입니다. 먼저 상태 준비(H·CX), 기저 변환(RY), 측정(`measure`) 세 부분만 찾으세요. `measured=False`는 측정을 붙이지 않아 이론값을 계산할 수 있게 합니다. 네 설정의 결과는 다음 반복문에서 별도로 실행하고 모읍니다.

In [ ]:
def chsh_circuit(x, y, bob_angle=np.pi/4, entangled=True, measured=True):
    qc = QuantumCircuit(2, 2) if measured else QuantumCircuit(2)
    if entangled:
        qc.h(0)
        qc.cx(0, 1)
    # entangled=False일 때는 비교용 곱상태 |00>에서 시작합니다.
    qc.barrier()
    alice_angle = (0.0, np.pi/2)[x]
    bob_setting = (bob_angle, -bob_angle)[y]
    qc.ry(-alice_angle, 0)
    qc.ry(-bob_setting, 1)
    if measured:
        qc.measure([0,1], [0,1])
    return qc

# x=1, y=0 설정의 회로를 먼저 확인합니다.
show_circuit(chsh_circuit(1, 0))

### 5.2 counts에서 상관관계 계산하기

출력 문자열은 `b a` 순서입니다. `00`, `11`의 횟수를 더하고 `01`, `10`의 횟수를 빼서 전체 shots로 나누면 $E_{xy}$의 추정값을 얻습니다. 먼저 이 계산을 함수로 작성합니다.

In [ ]:
def correlation(counts):
    return sum((1 if bit_at(bits,0)==bit_at(bits,1) else -1)*n
               for bits,n in counts.items()) / sum(counts.values())

네 가지 설정을 각각 4096 shots 실행합니다. 같은 상태를 네 번 이어서 측정하는 것이 아니라, **매 시행마다 Bell 상태를 새로 준비**하고 지정한 방향으로 측정합니다.

In [ ]:
SHOTS_CHSH = 4096        # 각 측정 설정당 shots; 총 시행 수는 4배입니다.
chsh_data, chsh_exact, chsh_rows = {}, {}, []
for i, (x,y) in enumerate(itertools.product([0,1], repeat=2)):
    counts = sample_counts(chsh_circuit(x,y), shots=SHOTS_CHSH, seed=SEED+i)
    chsh_data[x,y] = counts
    exact_state = Statevector.from_instruction(chsh_circuit(x,y,measured=False))
    chsh_exact[x,y] = float(np.real(exact_state.expectation_value(Pauli("ZZ"))))
    chsh_rows.append([x,y,str(counts),f"{correlation(counts):+.4f}",f"{chsh_exact[x,y]:+.4f}"])

이제 $E_{00},E_{01},E_{10}$은 더하고 $E_{11}$은 빼서 $S$를 구합니다. 비교용 이론값은 측정 전 회로의 상태벡터에서 계산했습니다.

In [ ]:
E = {xy: correlation(c) for xy,c in chsh_data.items()}
S_sample = E[0,0]+E[0,1]+E[1,0]-E[1,1]
S_exact = chsh_exact[0,0]+chsh_exact[0,1]+chsh_exact[1,0]-chsh_exact[1,1]
# 독립 설정별 표본평균의 분산을 더한, 큰 N에서의 근사 표준오차입니다.
se_S = np.sqrt(sum((1-E[xy]**2)/SHOTS_CHSH for xy in E))
table(["x", "y", "counts (b a)", "E sampled", "E exact"], chsh_rows)
print(f"S = {S_sample:.4f}  (approx. SE = {se_S:.4f})")
print(f"Exact S = {S_exact:.6f};  2*sqrt(2) = {2*np.sqrt(2):.6f}")
print("총 shots:", 4*SHOTS_CHSH)
fig, ax = plt.subplots(figsize=(7,3.2))
ax.bar(["E00","E01","E10","E11"], [E[xy] for xy in E])
ax.scatter(range(4), [chsh_exact[xy] for xy in E], color="#E58A32", s=55, label="Exact")
ax.axhline(0,color="gray",lw=.8)
ax.set(ylim=(-1.1,1.1), ylabel="Correlation")
ax.legend()
plt.show()

유한 shots의 $S$는 이론값 근처에서 흔들립니다. 표본값이 잠깐 $2\sqrt2$를 넘더라도 양자이론의 한계를 넘은 것이 아닙니다. 위 표준오차는 교육용 근사이며, 실제 Bell 검증의 유의확률 계산이나 실험 허점 분석을 대신하지 않습니다.

### 5.3 같은 데이터를 CHSH 게임으로 읽기

$x,y$를 균등하게 고르고, $a\oplus b=xy$이면 이긴다고 합시다. $xy=00,01,10$에서는 두 결과가 같아야 하고, $xy=11$에서는 달라야 합니다.

고전적 결정론 전략 16개를 전부 확인하면 최대 승률은 $3/4$입니다. 공유 난수를 쓰는 전략은 이러한 전략들의 혼합이므로 최대값을 높이지 못합니다. 위에서 정한 부호에 따르면

$$p_{\mathrm{win}}=\frac12+\frac{S}{8},\qquad
p_{\mathrm{win}}^{\mathrm{quantum}}=\cos^2(\pi/8)\simeq0.8536.$$

In [ ]:
classical_scores = []
for a0,a1,b0,b1 in itertools.product([0,1], repeat=4):
    wins = sum((((a0,a1)[x] ^ (b0,b1)[y]) == (x & y))
               for x,y in itertools.product([0,1],repeat=2))
    classical_scores.append(wins/4)

win_by_setting = []
for (x,y), counts in chsh_data.items():
    wins = sum(n for bits,n in counts.items()
               if (bit_at(bits,0) ^ bit_at(bits,1)) == (x & y))
    win_by_setting.append(wins/sum(counts.values()))
win_quantum = float(np.mean(win_by_setting))
print("고전 전략 16개의 최대 승률:", max(classical_scores))
print(f"샘플로 얻은 양자 전략 승률: {win_quantum:.4%}")
print(f"S에서 다시 계산한 승률: {0.5+S_sample/8:.4%}")
print(f"양자 전략의 이론 승률: {np.cos(np.pi/8)**2:.4%}")

### 5.4 각도를 바꾸면? · 선택 탐색

Bob의 두 방향을 $+\beta,-\beta$로 바꾸어 봅니다. Bell 상태에서는 $S(\beta)=2(\cos\beta+\sin\beta)$이며 $\beta=\pi/4$에서 최대가 됩니다. 같은 방향 설정에서 곱상태 $|00\rangle$의 결과도 비교합니다.

In [ ]:
beta_values = np.linspace(0, np.pi/2, 13)
scan_exact, scan_sample, scan_product = [], [], []
for i,beta in enumerate(beta_values):
    sampled, exact, product = [], [], []
    for j,(x,y) in enumerate(itertools.product([0,1],repeat=2)):
        sampled.append(correlation(sample_counts(chsh_circuit(x,y,beta), shots=1024, seed=SEED+4*i+j)))
        for entangled, values in [(True,exact),(False,product)]:
            sv = Statevector.from_instruction(chsh_circuit(x,y,beta,entangled=entangled,measured=False))
            values.append(float(np.real(sv.expectation_value(Pauli("ZZ")))))
    signs = np.array([1,1,1,-1])
    scan_sample.append(np.dot(signs,sampled))
    scan_exact.append(np.dot(signs,exact))
    scan_product.append(np.dot(signs,product))
fig, ax = plt.subplots(figsize=(8,3.5))
ax.plot(beta_values/np.pi,scan_exact,label="Bell: exact")
ax.scatter(beta_values/np.pi,scan_sample,s=25,label="Bell: 1024 shots / setting")
ax.plot(beta_values/np.pi,scan_product,label="Product |00>: exact")
ax.axhline(2,color="gray",ls="--",label="Local bound")
ax.axvline(.25,color="#E58A32",ls=":")
ax.set(xlabel=r"Bob angle $\beta/\pi$",ylabel="S",xticks=[0,.125,.25,.375,.5])
ax.legend(fontsize=9)
plt.show()

**확인하기:** $E_{11}$을 더하는 대신 빼는 이유를 게임의 승리 조건으로 설명해 보세요. 결과 문자열 `01`은 Alice와 Bob의 결과가 같다는 뜻일까요, 다르다는 뜻일까요?

<a id="teleportation"></a>
## Part 6. 양자 텔레포테이션

Alice의 입력 큐비트에 있는 상태 $|\psi\rangle$를 Bob의 큐비트로 옮깁니다. 미리 공유한 Bell 쌍 하나와 고전 비트 두 개가 필요합니다.

| 회로의 큐비트 | 소유자 | 역할 |
|:--|:--|:--|
| $q_0$ | Alice | 보낼 상태 $\vert \psi\rangle$ |
| $q_1$ | Alice | Bell 쌍의 절반 |
| $q_2$ | Bob | Bell 쌍의 나머지 절반, 최종 출력 |

진행 순서는 **Bell 쌍 준비 → Alice의 Bell 측정 → 두 결과 전달 → Bob의 조건부 보정**입니다. 코드의 `if_test`가 고전 측정 결과에 따른 보정을 표현합니다.

시뮬레이션에서는 검증을 위해 입력 상태를 알고 있습니다. 텔레포테이션 프로토콜의 게이트와 보정 규칙에는 그 상태의 각도를 넣을 필요가 없습니다.

### 6.1 입력 상태 준비

위상까지 전달되는지 확인하기 위해 $|0\rangle$나 $|+\rangle$ 대신 일반적인 상태를 사용합니다. 아래 `theta_in`, `phi_in`을 바꾸어 보세요.

In [ ]:
theta_in = 1.1
phi_in = 0.7
input_prep = QuantumCircuit(1, name="U_in")
input_prep.ry(theta_in, 0)
input_prep.rz(phi_in, 0)
input_state = Statevector.from_instruction(input_prep)
show_circuit(input_prep)
display(input_state.draw("latex"))
show_bloch([input_state], ["Input state"])

### 6.2 회로 작성

Alice는 $q_0\to q_1$에 CNOT, $q_0$에 $H$를 적용한 뒤 두 큐비트를 측정합니다. $q_0$의 결과를 $m_0=c_0$, $q_1$의 결과를 $m_1=c_1$로 기록합니다.

Bob의 보정 전 상태는 $X^{m_1}Z^{m_0}|\psi\rangle$입니다. 따라서 **$X$를 먼저 조건부 적용하고, 그다음 $Z$를 조건부 적용**하면 입력 상태가 복원됩니다.

| $m_0$ | $m_1$ | Bob의 보정 전 상태 | 시간 순서대로 가할 게이트 |
|:--:|:--:|:--|:--|
| 0 | 0 | $\vert \psi\rangle$ | 없음 |
| 0 | 1 | $X\vert \psi\rangle$ | $X$ |
| 1 | 0 | $Z\vert \psi\rangle$ | $Z$ |
| 1 | 1 | $XZ\vert \psi\rangle$ | $X$ 다음 $Z$ |

표는 $m_0,m_1$ 순서입니다. 반면 counts 문자열은 뒤에서부터 `c0`, `c1`, `c2`를 읽습니다.

**측정 결과에 따라 왜 $X$와 $Z$가 생길까요?**

$|\psi\rangle=\alpha|0\rangle+\beta|1\rangle$라고 합시다. 아래 식도 Qiskit과 같은 **$q_2q_1q_0$ 순서**를 사용합니다. 초기 상태는 $|\Phi^+\rangle_{q_2q_1}\otimes|\psi\rangle_{q_0}$입니다.

Alice가 $\mathrm{CX}_{0\to1}$을 적용하면 전체 상태는

$$\frac{\alpha|000\rangle+\beta|011\rangle+\alpha|110\rangle+\beta|101\rangle}{\sqrt2}$$

가 됩니다. 이어서 $q_0$에 $H$를 적용하고 Alice의 비트 $q_1q_0$별로 모으면

$$\begin{aligned}
|\Omega\rangle=\frac12\big(&|\psi\rangle_{q_2}\otimes|00\rangle_{q_1q_0}
+Z|\psi\rangle_{q_2}\otimes|01\rangle_{q_1q_0}\\
&+X|\psi\rangle_{q_2}\otimes|10\rangle_{q_1q_0}
+XZ|\psi\rangle_{q_2}\otimes|11\rangle_{q_1q_0}\big).
\end{aligned}$$

오른쪽 두 비트를 측정하면 해당 항이 선택됩니다. 예를 들어 결과 문자열이 `01`이면 $m_1=0,m_0=1$이고, Bob에게는 $Z|\psi\rangle$가 남습니다. $Z$를 다시 적용하면 $Z^2=I$이므로 원래 상태로 돌아옵니다. 나머지 분기도 같은 방식으로 보정표를 얻습니다.

네 Bob 상태는 모두 정규화되어 있고 앞의 계수는 $1/2$이므로, 각 측정 결과의 확률은 $1/4$입니다. 이 두 비트의 분포에는 $\alpha,\beta$가 드러나지 않지만, **미리 공유한 얽힘과 두 비트에 따른 보정**을 함께 사용하면 Bob의 큐비트에 상태를 복원할 수 있습니다.

**Python 조건문과 회로의 조건부 연산:** `if corrections:`는 Python이 회로를 만들 때 보정 블록을 넣을지 결정합니다. `with qc.if_test(...):`는 각 shot에서 얻은 고전 측정 결과에 따라 회로 안의 게이트를 실행합니다. 둘은 판단하는 시점이 다릅니다. 먼저 `m1 → X`, `m0 → Z`의 연결을 확인하세요.

In [ ]:
def teleport_circuit(preparation, corrections=True):
    qc = QuantumCircuit(3, 3)
    qc.compose(preparation, qubits=[0], inplace=True)  # 검증할 입력 준비
    qc.barrier()

    # 1. Alice(q1)와 Bob(q2)이 Bell 쌍을 공유합니다.
    qc.h(1)
    qc.cx(1, 2)
    qc.barrier()

    # 2. Alice의 Bell 측정
    qc.cx(0, 1)
    qc.h(0)
    qc.measure(0, 0)                # m0 -> c0
    qc.measure(1, 1)                # m1 -> c1
    qc.barrier()

    # 3. Alice의 고전 비트 두 개를 이용한 Bob의 보정
    if corrections:
        with qc.if_test((qc.clbits[1], 1)):
            qc.x(2)
        with qc.if_test((qc.clbits[0], 1)):
            qc.z(2)
    return qc                      # c2는 이후 Bob 검증에 사용합니다.

teleport = teleport_circuit(input_prep)
show_circuit(teleport)

이 회로는 중간 측정과 조건부 연산을 포함하므로 `Statevector.from_instruction(teleport)`로 최종 상태를 한 번에 계산하지 않습니다. 중간 측정의 결과에 따라 이후 연산이 달라지는 **동적 회로**를 지원하는 `AerSimulator`로 실행합니다.

### 6.3 입력 준비를 되돌려 검증하기

입력 준비 연산을 $U$라고 하면 $|\psi\rangle=U|0\rangle$입니다. Bob의 상태가 입력과 같다면 $U^\dagger$를 가했을 때 $|0\rangle$로 돌아와야 합니다. `preparation.inverse()`가 이 역연산을 만듭니다.

입력과 출력이 얼마나 같은지 나타내는 수를 **fidelity**라고 합니다. 순수한 목표 상태 $|\psi\rangle$에 대해

$$F=\langle\psi|\rho_B|\psi\rangle
=P(\text{Bob의 검증 결과가 }0)$$

이며 $0\le F\le1$입니다. 목표 상태와 같으면 1, 목표 상태와 직교하면 0입니다. 출력도 순수상태 $|\varphi\rangle$이면 $F=|\langle\psi|\varphi\rangle|^2$입니다.

$U^\dagger$와 마지막 측정은 **전달이 잘 되었는지 확인하는 검증용 연산**입니다. 텔레포테이션 자체에는 필요하지 않습니다. 일반 Z 측정 확률만 비교하는 것과 달리 상대위상이 잘못 전달된 경우도 검사할 수 있습니다.

In [ ]:
def verify_teleport(preparation, corrections=True, shots=2048):
    circuit = teleport_circuit(preparation, corrections=corrections)
    circuit.barrier()
    circuit.compose(preparation.inverse(), qubits=[2], inplace=True)
    circuit.measure(2, 2)
    counts = sample_counts(circuit, shots=shots)
    bob_counts = marginal_counts(counts, 2)
    fidelity_estimate = bob_counts["0"] / shots
    return circuit, counts, fidelity_estimate

같은 입력에 대해 보정을 적용한 경우와 생략한 경우를 각각 실행합니다. 여기서 표시되는 `F 추정`은 Bob의 검증 결과가 0인 빈도입니다.

In [ ]:
teleport_test, teleport_counts, F_corrected = verify_teleport(input_prep, corrections=True)
_, no_correction_counts, F_uncorrected = verify_teleport(input_prep, corrections=False)
show_circuit(teleport_test)
print("보정 있음 counts (Bob-test m1 m0):", teleport_counts)
print("보정 없음 counts (Bob-test m1 m0):", no_correction_counts)
print(f"보정 있음: F 추정 = {F_corrected:.4f}")
print(f"보정 없음: F 추정 = {F_uncorrected:.4f} (모든 분기 평균의 이론값 0.5)")

### 직접 바꾸기 · 함수 옵션 True와 False

`corrections` 한 값만 바꾸어 실행합니다. `True`에서는 검증 fidelity가 1, `False`에서는 유한 shots의 변동을 포함해 약 0.5입니다. 회로에서 조건부 X·Z 블록도 함께 달라지는지 확인하세요. 함수가 돌려주는 세 값을 `circuit, counts, fidelity`에 각각 받는 문법을 **여러 값 대입**이라고 합니다.

In [ ]:
corrections = True
circuit, counts, fidelity = verify_teleport(input_prep, corrections=corrections)
print("Fidelity:", fidelity)
show_circuit(circuit)

보정했을 때는 검증 결과가 모두 0입니다. 보정을 생략하고 네 측정 결과를 모으면 이론적으로 $F=1/2$이며, 유한 shots의 추정값은 그 주변에서 흔들립니다.

Alice의 결과 $(m_0,m_1)$별로 나누어도 전달이 성공했는지 확인해 봅니다. counts 문자열의 왼쪽 비트는 Bob의 검증 결과, 오른쪽 두 비트는 `m1 m0`입니다.

In [ ]:
branch_sample_rows = []
for m0,m1 in itertools.product([0,1], repeat=2):
    selected = {b:n for b,n in teleport_counts.items() if bit_at(b,0)==m0 and bit_at(b,1)==m1}
    total = sum(selected.values())
    success = sum(n for b,n in selected.items() if bit_at(b,2)==0)
    branch_sample_rows.append([m0,m1,total,f"{success/total:.4f}" if total else "no shots"])
table(["m0", "m1", "branch shots", "P(Bob-test=0 | branch)"], branch_sample_rows)

이상적인 시뮬레이터에서 보정 후의 실패 확률은 0이므로, 이 검증 결과는 shots가 유한해도 모두 0으로 나옵니다. 다만 실제 장치에서 관측 실패가 0회라는 사실만으로 실제 오류율이 정확히 0이라고 결론 내릴 수는 없습니다.

### 6.4 네 분기의 상태를 직접 확인하기 · 선택

이번에는 Alice의 측정 직전 상태를 계산한 뒤, 네 결과에 각각 투영해 Bob의 상태를 구합니다. 각 분기의 확률은 1/4이고, 보정 후의 fidelity는 모두 1이어야 합니다. 이는 표본 오차 없는 상태벡터 계산입니다.

아래 인덱스 `4*b + 2*m1 + m0`는 기저가 $|q_2q_1q_0\rangle$ 순서라는 데서 나옵니다. Bob의 비트 $b=0,1$만 남기고 Alice의 두 비트는 고정합니다.

In [ ]:
before_alice_measurement = QuantumCircuit(3)
before_alice_measurement.compose(input_prep, qubits=[0], inplace=True)
before_alice_measurement.h(1)
before_alice_measurement.cx(1, 2)
before_alice_measurement.cx(0, 1)
before_alice_measurement.h(0)
full_state = Statevector.from_instruction(before_alice_measurement)

branch_rows, branch_probabilities, branch_before, branch_after = [], [], [], []
for m0,m1 in itertools.product([0,1], repeat=2):
    amplitudes = np.array([full_state.data[4*b+2*m1+m0] for b in [0,1]])
    probability = float(np.vdot(amplitudes, amplitudes).real)
    bob_before = Statevector(amplitudes / np.sqrt(probability))
    correction = QuantumCircuit(1)
    if m1:
        correction.x(0)
    if m0:
        correction.z(0)
    bob_after = bob_before.evolve(correction)
    branch_probabilities.append(probability)
    branch_before.append(bob_before)
    branch_after.append(bob_after)
    branch_rows.append([m0,m1,f"{probability:.4f}",
                       f"{state_fidelity(input_state,bob_before):.6f}",
                       f"{state_fidelity(input_state,bob_after):.6f}"])
table(["m0","m1","branch probability","F before","F after"], branch_rows)

rho_without_message = DensityMatrix(sum(p*DensityMatrix(s).data for p,s in zip(branch_probabilities,branch_before)))
rho_after_correction = DensityMatrix(sum(p*DensityMatrix(s).data for p,s in zip(branch_probabilities,branch_after)))
show_bloch([input_state, rho_without_message, rho_after_correction],
           ["Input", "Bob without message: I/2", "Bob after correction"])
print("보정 전, 결과를 모른 Bob의 상태:")
display(array_to_latex(rho_without_message.data))

고전 비트를 받지 못한 Bob의 상태는 입력과 무관한 $I/2$입니다. 결과를 알게 되면 그 분기에 맞는 보정을 가할 수 있습니다. Alice의 입력 큐비트에는 원래 상태가 남지 않으므로 복제가 아니며, 고전 통신이 필요하므로 초광속 통신도 아닙니다.

**직접 바꾸기:** `theta_in`, `phi_in`을 바꾸고 Part 6의 셀들을 다시 실행하세요. 보정 후에는 모든 분기에서 같은 입력 상태가 복원되지만, 보정 전 fidelity는 입력과 분기에 따라 달라집니다.

### 연습 4. 측정 결과만 보고 보정하기

Part 6의 보정 표를 이용해 함수 안에 `X`와 `Z`를 추가하세요. 아래 코드는 각 분기에 대한 보정 전 상태를 새로 만들고, 네 가지 경우를 모두 검사합니다.

네 경우의 fidelity가 모두 1이면 목표를 만족합니다. [해설](#solution-4)

In [ ]:
def my_correction(m0, m1):
    correction = QuantumCircuit(1)
    # m1이 1이면 X, m0가 1이면 Z를 시간 순서대로 추가하세요.


    return correction

check_rows = []
for m0,m1 in itertools.product([0,1],repeat=2):
    error = QuantumCircuit(1)
    if m0:
        error.z(0)
    if m1:
        error.x(0)
    before = input_state.evolve(error)
    after = before.evolve(my_correction(m0,m1))
    check_rows.append([m0,m1,f"{state_fidelity(input_state,after):.6f}"])
table(["m0","m1","F after your correction"], check_rows)

<a id="primitives"></a>
## Part 7. 더 해보기 · 선택: Sampler와 Estimator

앞에서는 `AerSimulator.run()`으로 회로의 실행과 측정 과정을 드러내어 살펴보았습니다. Qiskit의 **V2 primitive 인터페이스**로도 같은 종류의 작업을 표현할 수 있습니다.

| 도구 | 입력 | 출력 |
|:--|:--|:--|
| `StatevectorSampler` | 마지막에 측정하는 회로 | 유한 shots의 bitstrings·counts |
| `StatevectorEstimator` | 유니터리 회로와 관측량 | 이 예제에서는 정확한 기댓값 |

둘 다 여기서는 로컬 시뮬레이션 구현입니다. 실제 장치의 Runtime primitive와 실행 환경은 다릅니다. 특히 `StatevectorSampler`는 중간 측정이 있는 텔레포테이션 회로를 실행하는 데 사용하지 않습니다.

In [ ]:
from qiskit.primitives import StatevectorSampler, StatevectorEstimator

primitive_bell = QuantumCircuit(2)
primitive_bell.h(0)
primitive_bell.cx(0, 1)
primitive_bell_m = primitive_bell.copy()
primitive_bell_m.measure_all()       # 고전 레지스터 이름은 meas

### 7.1 Sampler: 측정 결과 얻기

`run([회로])`의 결과에서 `[0]`은 첫 번째 회로의 결과를 선택합니다. `meas`는 `measure_all()`이 만든 고전 레지스터의 이름입니다. 다른 이름의 레지스터를 사용했다면 그 이름으로 읽습니다.

In [ ]:
sampler = StatevectorSampler(seed=SEED)
sampler_result = sampler.run([primitive_bell_m], shots=1024).result()[0]
print("Sampler counts:", sampler_result.data.meas.get_counts())

### 7.2 Estimator: 관측량의 기댓값 얻기

Estimator에는 **측정 명령을 붙이기 전 회로**와 관측량을 전달합니다. `SparsePauliOp`는 Pauli 연산자 또는 그 선형결합을 표현하는 클래스입니다.

아래 `StatevectorEstimator(default_precision=0.0)`는 이 유니터리 회로의 기댓값을 정확하게 계산합니다. 이 예제에는 유한 shots의 변동을 넣지 않습니다.

In [ ]:
estimator = StatevectorEstimator(default_precision=0.0)
observables = [SparsePauliOp(p) for p in ["ZZ", "XX", "YY"]]
estimator_result = estimator.run([(primitive_bell, observables)]).result()[0]
print("Estimator [<ZZ>, <XX>, <YY>]:", estimator_result.data.evs)

### 7.3 하나의 회로에 여러 각도 넣기

매번 회로 구조를 다시 만드는 대신 `Parameter`로 각도만 바꿀 수 있습니다. $R_y(\theta)$ 다음 CNOT을 가하면

$$|\psi(\theta)\rangle=\cos\frac{\theta}{2}|00\rangle
+\sin\frac{\theta}{2}|11\rangle,$$

따라서 $\langle ZZ\rangle=1$, $\langle XX\rangle=\sin\theta$, $\langle YY\rangle=-\sin\theta$입니다. `theta=0`에서는 곱상태, `theta=pi/2`에서는 Bell 상태입니다.

`parameter_values`의 모양은 `(25, 1)`입니다. 25개 각도를 넣으며 회로의 매개변수는 하나입니다. 관측량을 `(3, 1)` 형태로 전달하면 각 관측량을 모든 각도와 조합하여 결과가 **`(3, 25)`**가 됩니다. 행은 ZZ·XX·YY, 열은 각도에 대응합니다.

다음 셀은 필요한 import와 `estimator` 생성을 포함하므로, 패키지가 설치되어 있다면 **이 셀만 실행해도** 결과를 확인할 수 있습니다.

**선택 심화 · 배열 모양:** `angles[:, None]`은 각도를 담은 1차원 배열에 새 축을 붙여 `(25, 1)`로 만듭니다. 25는 실험할 각도 수, 1은 회로 매개변수 수입니다. 이 셀의 관측량 목록은 `(3, 1)`이고, 브로드캐스팅한 결과는 `(3, 25)`입니다. 처음에는 출력의 행이 관측량, 열이 각도라는 점을 읽을 수 있으면 충분합니다.

In [ ]:
# 앞의 Sampler·Estimator 실습을 실행하지 않아도 이 셀만 실행할 수 있습니다.
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

estimator = StatevectorEstimator(default_precision=0.0)

theta_parameter = Parameter("theta")
parameter_circuit = QuantumCircuit(2)
parameter_circuit.ry(theta_parameter, 0)
parameter_circuit.cx(0,1)
angles = np.linspace(0,np.pi,25)
parameter_values = angles[:,None]  # 25개 각도, 회로의 매개변수는 1개
observable_grid = [[SparsePauliOp(p)] for p in ["ZZ","XX","YY"]]
parameter_result = estimator.run([(parameter_circuit, observable_grid, parameter_values)]).result()[0]
parameter_evs = parameter_result.data.evs
print("결과 배열의 모양 (관측량, 각도):", parameter_evs.shape)
circuit_figure = parameter_circuit.draw(output="mpl")
display(circuit_figure)
plt.close(circuit_figure)
fig, ax = plt.subplots(figsize=(8,3.4))
for p,values in zip(["ZZ","XX","YY"],parameter_evs):
    ax.plot(angles/np.pi, values, label=f"<{p}>")
ax.set(xlabel=r"$\theta/\pi$",ylabel="Expectation value",ylim=(-1.1,1.1))
ax.legend()
plt.show()

<a id="vqa"></a>
## Part 8. VQA: 에너지를 최소화하는 회로 찾기

앞에서는 각도를 직접 정해 회로를 실행했습니다. 이제 결과가 좋아지는 방향으로 **고전 컴퓨터가 각도를 갱신**하도록 해 봅니다. 매개변수 양자 회로와 고전 최적화가 반복해서 정보를 주고받는 알고리즘을 VQA(Variational Quantum Algorithm, 변분 양자 알고리즘)라고 합니다.

진행 순서는 **각도 선택 → 회로로 상태 준비 → 비용함수 계산 → 고전 최적화기로 각도 갱신 → 반복**입니다. ‘비용함수’는 우리가 줄이고 싶은 값을 하나의 수로 나타낸 함수입니다. ‘Ansatz’는 매개변수를 바꾸며 탐색할 상태들을 정하는 회로 구조입니다.

이 실습에서는 에너지의 최솟값을 찾는 VQA인 **VQE(Variational Quantum Eigensolver)**의 작은 예제를 구현합니다. 양자 회로 부분도 로컬에서 시뮬레이션하므로, 여기서 실행 속도의 양자 우위를 보여 주는 것은 아닙니다.

### 8.1 무엇을 최소화할까요?

에너지를 나타내는 연산자인 해밀토니안을 다음과 같이 정합니다. $\mathcal H$는 Hadamard 게이트 $H$와 다른 기호입니다. 에너지 단위는 이 예제에서 1로 둡니다.

$$\mathcal H=-ZZ-XX,\qquad E(\theta)=\langle\psi(\theta)|\mathcal H|\psi(\theta)\rangle
=-\langle ZZ\rangle-\langle XX\rangle.$$

VQE는 이 기댓값을 줄여 가장 낮은 에너지의 상태를 찾습니다. 정규화된 임의의 상태는 해밀토니안의 고유상태들의 중첩으로 쓸 수 있고, 에너지 기댓값은 고유값들의 확률 가중 평균입니다. 따라서 최소 고유값 $E_0$보다 작아질 수 없습니다. 이것이 이 실습에 사용하는 **변분 원리**입니다.

회로는 $R_y(\theta)$ 다음 CNOT을 적용합니다.

$$|\psi(\theta)\rangle=\cos\frac\theta2|00\rangle+\sin\frac\theta2|11\rangle,
\qquad E(\theta)=-1-\sin\theta.$$

따라서 이 예제의 정답은 $\theta=\pi/2$에서 $E=-2$, 상태는 $|\Phi^+\rangle$입니다. 정답을 계산할 수 있는 작은 문제를 골라 최적화 과정이 제대로 작동하는지 확인합니다. 최적화 함수에는 아래에서 직접 얻은 기댓값을 전달합니다.

In [ ]:
from scipy.optimize import minimize
from qiskit.primitives import StatevectorEstimator

vqa_theta = Parameter("theta_vqa")
vqa_ansatz = QuantumCircuit(2)
vqa_ansatz.ry(vqa_theta, 0)
vqa_ansatz.cx(0, 1)
vqa_hamiltonian = SparsePauliOp.from_list([("ZZ", -1.0), ("XX", -1.0)])
vqa_estimator = StatevectorEstimator(default_precision=0.0)
show_circuit(vqa_ansatz)
print("해밀토니안:", vqa_hamiltonian)

### 8.2 회로에서 비용함수 계산하기

아래 함수는 각도를 받아 회로와 해밀토니안을 Estimator에 전달하고, 계산된 에너지를 숫자 하나로 반환합니다. 최적화기는 이 반환값으로 다음 각도를 선택합니다. `vqa_history`에는 비용함수를 평가한 순서대로 각도와 에너지를 기록합니다.

첫 실습에서는 정확한 기댓값을 사용해 최적화 과정 자체를 살펴봅니다. `default_precision=0.0`인 이 계산에는 유한 shots의 변동이 없습니다.

**응용 실습 · 비용함수의 계약:** `vqa_cost([angle])`는 각도 리스트를 받아 **실수 하나인 에너지**를 반환합니다. `parameters[0]`은 첫 번째 각도를 꺼내는 표현입니다. `.append((angle, energy))`는 변화 과정을 나중에 그리기 위해 기록을 추가합니다. `return energy`가 최적화기에 전달되는 값이며 `print`로 화면에 보여 주는 것과는 다릅니다.

In [ ]:
vqa_history = []

def vqa_cost(parameters):
    angle = float(parameters[0])
    result = vqa_estimator.run([
        (vqa_ansatz, vqa_hamiltonian, [angle])
    ]).result()[0]
    energy = float(np.asarray(result.data.evs).item())
    vqa_history.append((angle, energy))
    return energy

print("theta=0.25에서 계산한 에너지:", vqa_cost([0.25]))

**직접 바꾸기:** 아래 함수 호출의 `0.25`를 `np.pi/2`로 바꾸어 실행하면 에너지가 −2가 됩니다. 여기서는 사람이 각도를 고릅니다. 다음 셀에서는 `minimize`가 같은 비용함수를 여러 번 호출하며 각도를 고릅니다.

In [ ]:
print("Energy:", vqa_cost([0.25]))

### 8.3 고전 최적화기와 반복하기

SciPy의 `minimize`가 위 함수를 여러 번 호출합니다. 여기서는 매끄러운 비용함수에 사용할 수 있는 BFGS 방법을 사용합니다. 미분 함수를 별도로 전달하지 않았으므로 최적화기가 가까운 각도들의 에너지 차이로 기울기를 추정합니다.

`x0`는 출발 각도, `gtol`은 종료 판단에 쓰는 기울기 허용오차, `maxiter`는 최대 반복 횟수입니다. 매 반복에서 비용함수를 여러 번 호출할 수 있으므로 **비용함수 평가 횟수와 최적화 반복 횟수는 다릅니다.**

**함수를 인수로 전달하기:** `minimize(vqa_cost, ...)`에서는 함수 이름을 전달합니다. `vqa_cost([0.25])`처럼 지금 한 번 호출하는 표현과 구분하세요. 최적화기가 여러 각도를 넣어 함수를 호출합니다. `x0`는 출발점, `method`는 최적화 방법, `options`는 옵션을 담은 딕셔너리입니다. 첫 실습에서는 출발 각도와 최종 에너지, 평가 횟수를 읽는 데 집중하세요.

In [ ]:
vqa_history = []                 # 다시 실행할 때 이전 기록을 지웁니다.
vqa_result = minimize(
    vqa_cost,
    x0=np.array([0.25]),
    method="BFGS",
    options={"gtol": 1e-6, "maxiter": 60},
)
vqa_best_angle = float(vqa_result.x[0])
print("종료 조건 충족:", vqa_result.success)
print("종료 메시지:", vqa_result.message)
print(f"찾은 각도 / pi: {vqa_best_angle / np.pi:.6f}")
print(f"최종 에너지: {vqa_result.fun:.8f}")
print("최적화 반복:", vqa_result.nit, "| 비용함수 평가:", vqa_result.nfev)

vqa_records = np.asarray(vqa_history)
vqa_grid = np.linspace(-0.2, np.pi + 0.2, 200)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(vqa_grid / np.pi, -1 - np.sin(vqa_grid), label="Exact landscape")
axes[0].scatter(vqa_records[:, 0] / np.pi, vqa_records[:, 1], s=25,
                c=np.arange(len(vqa_records)), cmap="viridis", label="Evaluated points")
axes[0].set(xlabel=r"Angle $\theta/\pi$", ylabel="Energy")
axes[0].legend(fontsize=9)
axes[1].plot(np.arange(1, len(vqa_records)+1), vqa_records[:, 1], "o-", ms=4)
axes[1].axhline(-2, color="#E58A32", ls="--", label="Ground energy = -2")
axes[1].set(xlabel="Objective evaluations", ylabel="Energy")
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

최적화기가 종료했다고 해서 일반적인 문제에서 전역 최솟값을 찾았다는 보장이 생기는 것은 아닙니다. 이 예제에서는 행렬의 고유값을 직접 계산할 수 있으므로 정답과 비교합니다. 더 큰 문제에서는 이런 직접 계산이 어려워지는 것이 VQE를 연구하는 이유 중 하나입니다.

아래 fidelity는 찾은 상태와 정확한 바닥상태가 얼마나 같은지 나타냅니다. 회로가 충분한 상태를 표현하지 못하거나 최적화가 잘되지 않으면 최저 에너지에 도달하지 못할 수 있습니다.

In [ ]:
vqa_eigenvalues, vqa_eigenvectors = np.linalg.eigh(vqa_hamiltonian.to_matrix())
vqa_ground_state = Statevector(vqa_eigenvectors[:, 0])
vqa_best_circuit = vqa_ansatz.assign_parameters({vqa_theta: vqa_best_angle})
vqa_best_state = Statevector.from_instruction(vqa_best_circuit)
vqa_fidelity = state_fidelity(vqa_ground_state, vqa_best_state)
print("해밀토니안의 고유값:", vqa_eigenvalues)
print(f"정확한 최저 에너지: {vqa_eigenvalues[0]:.8f}")
print(f"정확한 바닥상태와의 fidelity: {vqa_fidelity:.8f}")
display(vqa_best_state.draw("latex"))

### 8.4 유한 shots로 에너지를 구하면? · 선택

실제 측정에서는 $\langle ZZ\rangle$와 $\langle XX\rangle$를 횟수에서 추정합니다. XX 측정은 두 큐비트에 각각 $H$를 적용하고 Z 측정하는 방식입니다. 두 설정의 상관관계를 각각 구한 뒤 부호를 붙여 더하면 에너지 추정값이 됩니다.

여기서는 최적화 과정 중의 한 각도 $\theta=0.7$을 여러 번 평가합니다. 이 문제의 정확한 최적점에서는 ZZ와 XX가 모두 확정적인 값을 주므로, 그 점만 보면 유한 shots의 변동을 관찰하기 어렵기 때문입니다.

In [ ]:
def vqa_energy_from_counts(angle, shots=256, seed=SEED):
    bound = vqa_ansatz.assign_parameters({vqa_theta: float(angle)})
    correlations = []
    for index, basis in enumerate(["Z", "X"]):
        measured = bound.copy()
        if basis == "X":
            measured.h(0)
            measured.h(1)
        measured.measure_all()
        counts = sample_counts(measured, shots=shots, seed=seed+index)
        value = sum((1 if bit_at(bits, 0) == bit_at(bits, 1) else -1) * count
                    for bits, count in counts.items()) / shots
        correlations.append(value)
    return -sum(correlations)

vqa_probe_angle = 0.7
vqa_sample_rows = []
for shots in [64, 256, 1024]:
    estimates = [vqa_energy_from_counts(vqa_probe_angle, shots, SEED+10*i)
                 for i in range(5)]
    vqa_sample_rows.append([shots, 2*shots, ", ".join(f"{x:.4f}" for x in estimates)])
table(["shots / basis", "total shots / evaluation", "5 energy estimates"], vqa_sample_rows)
print(f"theta=0.7의 정확한 에너지: {-1-np.sin(vqa_probe_angle):.6f}")

유한 shots와 장치 잡음이 있으면 최적화기가 받는 에너지에도 오차가 생깁니다. 위의 정확한 에너지용 최적화 설정을 그대로 사용해도 항상 잘 작동한다고 기대할 수는 없습니다. 에너지 평가에 필요한 shots와 최적화의 반복 횟수도 계산 비용에 포함됩니다.

### 연습 5. Ansatz에서 CNOT을 빼면?

`RY(theta)`만 q0에 적용하고 q1을 $|0\rangle$로 두면, 같은 해밀토니안의 에너지 $-2$에 도달할 수 있을까요? $\langle ZZ\rangle$와 $\langle XX\rangle$를 각각 생각해 보세요. [해설](#solution-5)

다음 셀은 각도별 에너지를 계산합니다. 최적화 전에 회로가 표현할 수 있는 상태의 범위를 점검하는 연습입니다.

In [ ]:
exercise_5_energies = []
exercise_5_angles = np.linspace(0, np.pi, 41)
for angle in exercise_5_angles:
    exercise_5 = QuantumCircuit(2)
    exercise_5.ry(float(angle), 0)    # CNOT이 없는 ansatz
    trial = Statevector.from_instruction(exercise_5)
    exercise_5_energies.append(float(np.real(trial.expectation_value(vqa_hamiltonian))))
print("이 회로에서 스캔한 최소 에너지:", min(exercise_5_energies))
print("전체 해밀토니안의 최저 에너지:", vqa_eigenvalues[0])

<a id="qec"></a>
## Part 9. QEC: 오류를 검출하고 상태 복원하기

Part 6에서는 측정 결과에 따라 $X,Z$를 적용해 상태를 복원했습니다. 이번에는 오류에 대한 정보를 측정해 적절한 보정을 선택합니다. 이를 **양자 오류 정정(QEC, Quantum Error Correction)**이라고 합니다.

실습은 **인코딩 → 오류 발생 → 신드롬 측정 → 조건부 보정 → 검증** 순서로 진행합니다. 먼저 데이터 큐비트에 **최대 한 번의 $X$ 오류**가 발생하고, 인코딩·측정·보정 게이트는 이상적이라고 가정합니다. 뒤에서 이 가정이 깨지면 어떻게 되는지도 확인합니다.

### 9.1 하나의 논리 큐비트를 세 물리 큐비트에 인코딩하기

보호하려는 상태가 $|\psi\rangle=\alpha|0\rangle+\beta|1\rangle$일 때, 보조 데이터 큐비트 두 개를 $|0\rangle$로 준비하고 CNOT 두 개를 적용합니다.

$$|0\rangle_{q_2}|0\rangle_{q_1}(\alpha|0\rangle+\beta|1\rangle)_{q_0}
\longmapsto\alpha|000\rangle+\beta|111\rangle
=\alpha|0_L\rangle+\beta|1_L\rangle.$$

$|0_L\rangle=|000\rangle$, $|1_L\rangle=|111\rangle$를 논리 기저라고 부릅니다. 하나의 논리 큐비트의 정보를 세 물리 큐비트에 걸쳐 저장합니다. 결과는 일반적으로 $|\psi\rangle^{\otimes3}$과 다르므로, 알 수 없는 상태를 세 개로 복제한 것이 아닙니다.

In [ ]:
from qiskit import ClassicalRegister
from qiskit_aer.noise import pauli_error

qec_prep = QuantumCircuit(1)
qec_prep.ry(1.1, 0)
qec_prep.rz(0.7, 0)              # 중첩과 상대위상을 포함한 입력
qec_input_state = Statevector.from_instruction(qec_prep)

qec_encoder = QuantumCircuit(3)
qec_encoder.cx(0, 1)
qec_encoder.cx(0, 2)
qec_encoded = QuantumCircuit(3)
qec_encoded.compose(qec_prep, [0], inplace=True)
qec_encoded.compose(qec_encoder, inplace=True)
show_circuit(qec_encoded)
display(Statevector.from_instruction(qec_encoded).draw("latex"))

### 9.2 상태 자체가 아니라 오류의 흔적 측정하기

예를 들어 q1에 $X$ 오류가 생기면 $\alpha|010\rangle+\beta|101\rangle$가 됩니다. 데이터 세 개를 직접 Z 측정하면 어느 비트가 다른지는 알 수 있지만, $\alpha,\beta$가 담긴 중첩은 유지되지 않습니다.

대신 이웃한 비트가 **같은지 다른지**만 측정합니다. 두 비트가 같으면 패리티가 0, 다르면 1입니다. 보조 큐비트를 $|0\rangle$로 준비하고 두 데이터 큐비트에서 CNOT을 걸면, 그 보조 큐비트에 두 비트의 XOR가 기록됩니다.

$$s_{01}=q_0\oplus q_1,\qquad s_{12}=q_1\oplus q_2.$$

이 식의 $q_i$는 각 계산 기저 성분의 비트를 뜻합니다. 실제로는 데이터 비트를 직접 읽지 않고 보조 큐비트를 측정합니다. 관측량으로 쓰면 각각 $Z_0Z_1$, $Z_1Z_2$의 측정이며, 패리티 0은 고윳값 $+1$, 패리티 1은 $-1$에 대응합니다. 두 관측량은 서로 교환하므로 함께 측정할 수 있습니다.

$|000\rangle$와 $|111\rangle$는 모두 패리티가 `(0, 0)`입니다. q1에 오류가 난 $|010\rangle$와 $|101\rangle$는 모두 `(1, 1)`입니다. 따라서 **같은 오류를 겪은 두 성분을 구별하지 않으면서 오류 위치는 알아낼 수 있습니다.** 이 측정 결과를 신드롬(syndrome)이라고 합니다.

| 오류 가정 | $s_{01}$ | $s_{12}$ | counts의 `s12 s01` | 보정 |
|:--|:--:|:--:|:--:|:--|
| 오류 없음 | 0 | 0 | `00` | 없음 |
| q0의 $X$ | 1 | 0 | `01` | q0에 $X$ |
| q1의 $X$ | 1 | 1 | `11` | q1에 $X$ |
| q2의 $X$ | 0 | 1 | `10` | q2에 $X$ |

다음 회로에서 q0·q1·q2는 데이터, q3·q4는 신드롬 측정용 보조 큐비트입니다. q1에 오류를 넣으면 `11`만 나오는지 확인합니다.

In [ ]:
qec_syndrome_demo = QuantumCircuit(5, 2)
qec_syndrome_demo.compose(qec_encoded, [0, 1, 2], inplace=True)
qec_syndrome_demo.x(1)            # q1의 비트 반전 오류
qec_syndrome_demo.barrier()
qec_syndrome_demo.cx(0, 3)
qec_syndrome_demo.cx(1, 3)        # q3에 s01 기록
qec_syndrome_demo.cx(1, 4)
qec_syndrome_demo.cx(2, 4)        # q4에 s12 기록
qec_syndrome_demo.measure(3, 0)
qec_syndrome_demo.measure(4, 1)
show_circuit(qec_syndrome_demo)
qec_demo_counts = sample_counts(qec_syndrome_demo, shots=256)
print("신드롬 counts (s12 s01):", qec_demo_counts)

**직접 바꾸기:** 위 코드의 오류 삽입 줄을 `qec_syndrome_demo.x(0)` 또는 `.x(2)`로 바꾸고 재실행하세요. 회로 그림에서 X가 이동하고, 신드롬은 각각 `01`, `10`으로 바뀝니다. 원래 `.x(1)`에서는 `11`입니다. 출력 비트 순서는 `s12 s01`입니다. 단일 X 오류라는 가정 안에서만 이 표로 오류 위치를 판별할 수 있습니다.

### 9.3 신드롬에 맞추어 보정하기

`syndrome`이라는 2비트 고전 레지스터를 사용하면 `01`, `11`, `10`을 각각 정수 1, 3, 2로 읽을 수 있습니다. 그 값으로 `if_test`를 걸어 해당 큐비트에 $X$를 적용합니다. $X^2=I$이므로 한 번 발생한 비트 반전을 되돌립니다.

마지막에는 **인코딩된 상태 전체가 돌아왔는지** 검사합니다. 인코딩의 역연산으로 디코딩한 뒤 q0의 입력 준비도 되돌리면, 오류 없이 복원되었을 때 데이터 세 큐비트가 모두 $|000\rangle$이어야 합니다.

$$F_{\mathrm{code}}=\langle\psi_L|\rho_{\mathrm{data}}|\psi_L\rangle
=P(\text{검증용 데이터 측정 결과 }000).$$

이는 목표 인코딩 상태와의 fidelity입니다. 디코딩 후 q0만 검사하면 나머지 데이터 큐비트에 남은 오류를 놓칠 수 있으므로 세 개를 모두 확인합니다. 디코딩과 역준비는 검증 절차이며, 계속 정보를 보호하려면 보정 후 상태를 인코딩된 채로 유지합니다.

**응용 실습 · 오류 정정 회로 읽기:** 긴 함수를 한꺼번에 외우지 말고 입력 준비 → 인코딩 → 오류 삽입 → 신드롬 측정 → 보정 → 검증 순서로 찾으세요. 이 예제는 인코딩·측정·보정이 이상적이고 데이터에 최대 한 개의 X 오류가 있다는 가정에서 시작합니다. 임의의 오류를 모두 고치는 코드가 아닙니다.

In [ ]:
def bitflip_recovery(preparation, x_errors=(), z_errors=(), correction=True, p=None):
    syndrome = ClassicalRegister(2, "syndrome")
    check = ClassicalRegister(3, "check")
    qc = QuantumCircuit(5)
    qc.add_register(syndrome, check)

    # 1. 입력 준비와 인코딩: q0, q1, q2가 데이터입니다.
    qc.compose(preparation, [0], inplace=True)
    qc.cx(0, 1)
    qc.cx(0, 2)
    qc.barrier()

    # 2. 지정한 오류 또는 독립적인 확률 p의 X 오류를 넣습니다.
    for qubit in x_errors:
        qc.x(qubit)
    for qubit in z_errors:
        qc.z(qubit)
    if p is not None:
        channel = pauli_error([("X", p), ("I", 1-p)]).to_instruction()
        for qubit in range(3):
            qc.append(channel, [qubit])
    qc.barrier()

    # 3. 보조 큐비트 q3, q4로 패리티를 읽습니다.
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(1, 4)
    qc.cx(2, 4)
    qc.measure(3, syndrome[0])
    qc.measure(4, syndrome[1])

    # 4. 신드롬은 s12 s01 순서의 정수입니다.
    if correction:
        with qc.if_test((syndrome, 1)):
            qc.x(0)
        with qc.if_test((syndrome, 3)):
            qc.x(1)
        with qc.if_test((syndrome, 2)):
            qc.x(2)
    qc.barrier()

    # 5. 검증: 인코딩과 입력 준비를 되돌린 뒤 데이터 전체를 측정합니다.
    qc.cx(0, 2)
    qc.cx(0, 1)
    qc.compose(preparation.inverse(), [0], inplace=True)
    qc.measure([0, 1, 2], check)
    return qc


def qec_result_from_counts(counts):
    total = sum(counts.values())
    syndromes = {}
    success = 0
    for bits, count in counts.items():
        label = f"{bit_at(bits, 1)}{bit_at(bits, 0)}"
        syndromes[label] = syndromes.get(label, 0) + count
        if all(bit_at(bits, index) == 0 for index in [2, 3, 4]):
            success += count
    return syndromes, success / total

### 9.4 네 가지 경우를 실행하고 비교하기

오류 없음과 q0·q1·q2의 단일 $X$ 오류를 각각 실행합니다. 동일한 오류에 대해 보정했을 때와 보정하지 않았을 때의 $F_{\mathrm{code}}$ 추정값을 비교합니다. 여기서 신드롬 측정은 두 경우 모두 수행하고, 조건부 $X$ 보정만 켜거나 끕니다.

In [ ]:
qec_cases = [("No error", ()), ("X on q0", (0,)), ("X on q1", (1,)), ("X on q2", (2,))]
qec_rows, qec_fidelities = [], {}
for name, errors in qec_cases:
    values = []
    for corrected in [False, True]:
        circuit = bitflip_recovery(qec_prep, x_errors=errors, correction=corrected)
        counts = sample_counts(circuit, shots=1024)
        syndromes, fidelity = qec_result_from_counts(counts)
        values.append(fidelity)
    qec_fidelities[name] = values
    qec_rows.append([name, str(syndromes), f"{values[0]:.4f}", f"{values[1]:.4f}"])
table(["Error", "syndrome counts", "F_code without correction", "F_code with correction"], qec_rows)

qec_checked_circuit = bitflip_recovery(qec_prep, x_errors=(1,))
qec_figure = qec_checked_circuit.draw(output="mpl", fold=22)
display(qec_figure)
plt.close(qec_figure)

이 실습에서는 오류 없음과 세 가지 단일 $X$ 오류 모두 보정 후 $F_{\mathrm{code}}=1$입니다. 입력에 상대위상도 포함되어 있고 역준비 검증이 그 위상까지 검사하므로, 측정 비트의 분포만 맞춘 경우와 구별할 수 있습니다.

### 9.5 이 코드가 고치지 못하는 오류

**위상 오류:** 어느 한 데이터 큐비트에 $Z$가 작용하면 $\alpha|000\rangle-\beta|111\rangle$가 됩니다. 두 패리티는 여전히 0이어서 이 코드의 신드롬으로 검출되지 않습니다. 특히 논리 $|+\rangle$가 논리 $|-\rangle$로 바뀌면 두 상태는 직교합니다.

**두 개의 비트 반전:** q0와 q1에 $X$가 생기면 신드롬은 `10`입니다. 이는 q2의 단일 $X$와 같습니다. 단일 오류라고 가정한 보정기는 q2에도 $X$를 적용하여 결과적으로 세 큐비트가 모두 뒤집힙니다. 이는 논리 비트 반전 $X_L$입니다.

이 한계를 직접 확인합니다. 두 오류의 영향을 분명히 보기 위해 첫 경우에는 $|+\rangle$, 두 번째에는 $|0\rangle$를 입력합니다.

In [ ]:
qec_plus_prep = QuantumCircuit(1)
qec_plus_prep.h(0)
qec_zero_prep = QuantumCircuit(1)

qec_limit_cases = [
    ("Z on q0; input |+>", qec_plus_prep, (), (0,)),
    ("X on q0 and q1; input |0>", qec_zero_prep, (0, 1), ()),
]
qec_limit_rows = []
for label, prep, xs, zs in qec_limit_cases:
    counts = sample_counts(bitflip_recovery(prep, x_errors=xs, z_errors=zs), shots=512)
    syndromes, fidelity = qec_result_from_counts(counts)
    qec_limit_rows.append([label, str(syndromes), f"{fidelity:.4f}"])
table(["Error and input", "syndrome counts", "F_code after attempted correction"], qec_limit_rows)

### 9.6 오류 확률을 바꾸어 보기 · 선택

이번에는 세 데이터 큐비트 각각에 독립적으로 확률 $p$의 $X$ 오류가 생긴다고 가정합니다. 이는 $X$를 작은 각도로 회전시키는 것이 아니라, 각 위치에 확률적으로 $I$ 또는 $X$가 작용하는 잡음 채널입니다.

오류가 0개나 1개이면 복원되고, 2개나 3개이면 논리 $X$ 오류가 남습니다. 따라서 한 번의 이상적인 보정 후 논리 비트 반전 확률은

$$p_L={3\choose2}p^2(1-p)+p^3=3p^2-2p^3.$$

아래에서는 입력을 $|0\rangle$로 고정합니다. 이 경우 논리 $X$가 출력 $|1\rangle$로 구별되므로 검증 실패 빈도를 $p_L$과 비교할 수 있습니다. 일반 입력에서 fidelity는 항상 $1-p_L$인 것은 아닙니다. 예를 들어 $X$의 고유상태는 논리 $X$에 의해 물리적으로 바뀌지 않습니다.

이 모델에서는 $0<p<1/2$일 때 $p_L<p$입니다. 다만 인코딩·보조 큐비트·측정·보정 자체의 오류는 포함하지 않았으므로, 이 결과를 실제 장치의 오류 정정 임계값으로 해석해서는 안 됩니다.

In [ ]:
qec_noise_points = np.array([0.0, 0.05, 0.10, 0.20, 0.30, 0.40])
qec_observed_failures = []
for index, probability in enumerate(qec_noise_points):
    circuit = bitflip_recovery(qec_zero_prep, p=float(probability))
    counts = sample_counts(circuit, shots=4096, seed=SEED+index)
    _, fidelity = qec_result_from_counts(counts)
    qec_observed_failures.append(1-fidelity)

qec_probability_grid = np.linspace(0, 0.5, 200)
fig, ax = plt.subplots(figsize=(7.5, 3.7))
ax.plot(qec_probability_grid, qec_probability_grid, "--", color="gray", label="Unencoded: p")
ax.plot(qec_probability_grid, 3*qec_probability_grid**2-2*qec_probability_grid**3,
        label="Encoded: exact logical X probability")
ax.scatter(qec_noise_points, qec_observed_failures, color="#E58A32", s=35,
           label="Encoded: 4096 shots, input |0>")
ax.set(xlabel="Independent X error probability p", ylabel="Error probability", ylim=(-0.015, 0.52))
ax.legend(fontsize=9)
plt.show()

### 연습 6. 신드롬이 알려 주는 것

1. 신드롬 `01`이 나왔다면 어떤 보정을 해야 하나요? 여기서 적용한 오류 가정은 무엇인가요?
2. q1과 q2에 동시에 $X$가 발생했을 때도 `01`이 나오는지 확인하고, 보정 후 어떤 논리 연산이 남는지 설명해 보세요.
3. 보조 큐비트가 아니라 데이터 큐비트를 직접 Z 측정해서 오류를 찾으면 왜 중첩을 보호하지 못할까요?

[해설](#solution-6)

3큐비트 비트 반전 코드는 **한 번의 비트 반전**을 정정하는 입문 예제입니다. 임의의 단일 큐비트 오류를 정정하려면 위상 오류까지 다룰 수 있는 더 일반적인 코드가 필요합니다. 그런 코드와 반복 신드롬 측정, 오류가 있는 정정 회로의 설계는 다음 단계의 주제입니다.

## 세 응용의 연결

| 응용 | 회로에서 얻는 고전 정보 | 그 정보를 사용하는 방법 |
|:--|:--|:--|
| 텔레포테이션 | Alice의 두 측정 비트 | Bob에게 필요한 Pauli 보정 선택 |
| VQA | 비용함수의 기댓값 또는 그 추정값 | 다음 회로 매개변수를 고전 최적화기로 선택 |
| QEC | 오류 신드롬 | 데이터에 필요한 보정 선택 |

세 응용 모두 양자 회로와 고전 정보 처리가 함께 작동합니다. 다만 텔레포테이션과 QEC의 조건부 게이트는 **같은 시행 안에서의 측정 결과**를 사용하고, 여기서 구현한 VQA의 최적화기는 **회로 평가가 끝난 뒤 다음 실행의 각도**를 정합니다.

## 돌아보기

1. 상태벡터를 계산하는 것과 1 shot을 실행하는 것은 어떻게 다른가요?
2. $P(0)=P(1)=1/2$이면 상태가 반드시 $|+\rangle$인가요?
3. 0이 750회, 1이 250회 나온 Z 측정에서 $\langle Z\rangle$의 추정값은 얼마인가요?
4. CNOT을 $|00\rangle$에 적용해도 얽힘이 생기나요?
5. Bell 상태의 각 큐비트는 왜 Bloch 구면의 원점에 놓이나요?
6. CHSH의 네 상관관계를 어떤 부호로 합쳐야 하나요?
7. 텔레포테이션에서 고전 비트 두 개를 Bob에게 알려주지 않으면 어떻게 되나요?

8. VQA에서 양자 회로와 고전 최적화기는 각각 무엇을 계산하나요?
9. Ansatz에 정답 상태가 포함되지 않으면 최적화만으로 해결할 수 있나요?
10. QEC의 신드롬 측정이 입력 상태의 중첩을 유지할 수 있는 이유는 무엇인가요?
11. 이 3큐비트 코드가 위상 오류를 고치지 못하는 이유는 무엇인가요?

각 답을 설명하고, 위 실습에서 그 설명을 확인할 수 있는 출력을 찾아보세요.

<a id="solutions"></a>
## 연습문제 해설

직접 작성한 뒤 비교하세요. 정답은 하나만 있는 것이 아닙니다. 특히 전역위상만 다른 상태는 같은 물리적 상태를 나타냅니다.

<a id="solution-1"></a>
### 연습 1

$H|0\rangle=|+\rangle$을 만든 뒤 $S$로 $|1\rangle$ 성분에 $i$를 곱합니다. Bloch 벡터는 $+Y$ 방향입니다.

```python
exercise_1 = QuantumCircuit(1)
exercise_1.h(0)
exercise_1.s(0)
```

<a id="solution-2"></a>
### 연습 2

```python
exercise_2 = QuantumCircuit(1, 1)
exercise_2.h(0)
exercise_2.p(phase_ex2, 0)
exercise_2.h(0)
exercise_2.measure(0, 0)
```

$P(1)=\sin^2(\phi/2)$입니다. $\phi=\pi$이면 항상 1, $\phi=\pi/2$이면 0과 1이 각각 확률 1/2로 나옵니다.

<a id="solution-3"></a>
### 연습 3

```python
exercise_3 = QuantumCircuit(2)
exercise_3.h(0)
exercise_3.cx(0, 1)
exercise_3.x(0)
exercise_3.z(0)
```

Qiskit 순서에서 출력은 $(|10\rangle-|01\rangle)/\sqrt2$입니다. 목표와 전역위상 $-1$만 다르므로 `equiv`는 `True`를 반환합니다.

<a id="solution-4"></a>
### 연습 4

```python
def my_correction(m0, m1):
    correction = QuantumCircuit(1)
    if m1:
        correction.x(0)
    if m0:
        correction.z(0)
    return correction
```

보정 전 연산은 $X^{m_1}Z^{m_0}$, 보정 연산은 $Z^{m_0}X^{m_1}$입니다. 코드에서는 시간 순서대로 $X$ 다음 $Z$를 적습니다. 두 비트가 모두 1일 때 순서를 반대로 적용해도 전역위상만 달라지지만, 표와 코드에서는 위 순서를 일관되게 사용했습니다.


<a id="solution-5"></a>
### 연습 5

CNOT이 없으면 상태는 $|0\rangle_{q_1}\otimes R_y(\theta)|0\rangle_{q_0}$입니다. 따라서 $\langle ZZ\rangle=\cos\theta$, $\langle XX\rangle=\langle X\rangle_{q_1}\langle X\rangle_{q_0}=0$이고, 에너지는 $-\cos\theta$입니다. 최솟값은 $-1$이므로 정확한 바닥 에너지 $-2$에 도달하지 못합니다. 이 회로 구조에는 얽힌 바닥상태가 포함되지 않습니다.

<a id="solution-6"></a>
### 연습 6

1. 최대 한 개의 $X$ 오류라는 가정 아래 `01`은 q0의 오류이므로 q0에 $X$를 적용합니다.
2. q1과 q2의 두 오류도 같은 패리티를 만듭니다. q0에 잘못 보정하면 전체에는 $X_0X_1X_2=X_L$이 남아 $\alpha|0_L\rangle+\beta|1_L\rangle$를 $\beta|0_L\rangle+\alpha|1_L\rangle$로 바꿉니다.
3. 데이터의 Z 측정은 $|000\rangle$와 $|111\rangle$를 구별해 두 성분 사이의 결맞음을 없앱니다. 패리티 측정은 같은 오류를 겪은 두 성분에 같은 결과를 주므로, 그 성분들을 구별하지 않고 오류 정보를 얻습니다.

```python
# 두 비트 반전이 하나의 비트 반전과 같은 신드롬을 주는 예
exercise_6_counts = sample_counts(
    bitflip_recovery(qec_zero_prep, x_errors=(1, 2)), shots=512
)
print(qec_result_from_counts(exercise_6_counts))
# 신드롬은 '01', 이 입력에서 F_code는 0입니다.
```

### 복습 질문의 핵심

1. 상태벡터는 시뮬레이터가 계산한 진폭 전체입니다. 1 shot은 상태 준비부터 연산과 측정까지의 한 시행입니다.
2. 아닙니다. $|-\rangle$, $|+i\rangle$ 등도 같은 Z 측정 확률을 줍니다. 상대위상을 알려면 X·Y 측정 같은 추가 정보가 필요합니다.
3. 비트에 대응하는 관측값은 $+1,-1$이므로 $(750-250)/1000=0.5$입니다. 비트 자체의 평균 $0.25$와 다릅니다.
4. $\mathrm{CX}|00\rangle=|00\rangle$이므로 이 경우 얽힘이 없습니다.
5. 각 큐비트만의 상태가 $I/2$이고 세 Pauli 기댓값이 모두 0이기 때문입니다. 전체 Bell 상태는 여전히 순수상태입니다.
6. 같은 결과의 빈도에서 다른 결과의 빈도를 빼서 각 $E_{xy}$를 구한 뒤, $S=E_{00}+E_{01}+E_{10}-E_{11}$을 계산합니다.
7. 결과를 모르는 Bob의 상태는 $I/2$입니다. 두 비트가 있어야 분기에 맞는 보정을 선택할 수 있습니다.
8. 양자 회로 부분은 주어진 각도의 상태를 준비하고 비용에 필요한 기댓값을 제공합니다. 고전 최적화기는 그 값으로 다음 각도를 정합니다. 여기서는 양자 회로도 시뮬레이터로 실행합니다.
9. 해결할 수 없습니다. 최적화는 선택한 회로가 표현할 수 있는 상태들 안에서 탐색합니다.
10. 같은 오류를 겪은 논리 0·1 성분이 같은 패리티를 주기 때문에 입력의 확률진폭을 구별하지 않고 오류 정보를 얻습니다.
11. $Z$ 오류는 논리 상태의 상대 부호를 바꾸지만 두 Z 패리티는 바꾸지 않아 신드롬이 00으로 남습니다.


<a id="references"></a>
## 참고 자료

Python 문법과 Jupyter 조작은 함께 제공된 `python tutorial.ipynb`, `Tutorial on Jupyter Notebook.ipynb`에서 복습할 수 있습니다.

- [Qiskit 2.2: QuantumCircuit](https://quantum.cloud.ibm.com/docs/en/api/qiskit/2.2/qiskit.circuit.QuantumCircuit)
- [Qiskit 2.2: Statevector](https://quantum.cloud.ibm.com/docs/en/api/qiskit/2.2/qiskit.quantum_info.Statevector)
- [Qiskit: 비트 순서](https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering)
- [Qiskit 2.2: StatevectorSampler](https://quantum.cloud.ibm.com/docs/en/api/qiskit/2.2/qiskit.primitives.StatevectorSampler)
- [Qiskit 2.2: StatevectorEstimator](https://quantum.cloud.ibm.com/docs/en/api/qiskit/2.2/qiskit.primitives.StatevectorEstimator)
- [IBM Quantum Learning: 텔레포테이션과 CHSH의 Qiskit 구현](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/entanglement-in-action/qiskit-implementation)

- [IBM Quantum Learning: Variational algorithms](https://quantum.cloud.ibm.com/learning/en/courses/variational-algorithm-design/variational-algorithms)
- [IBM Quantum Learning: Optimization loops](https://quantum.cloud.ibm.com/learning/en/courses/variational-algorithm-design/optimization-loops)
- [IBM Quantum Learning: Repetition codes](https://quantum.cloud.ibm.com/learning/en/courses/foundations-of-quantum-error-correction/correcting-quantum-errors/repetition-codes)
- [IBM Quantum: 동적 회로를 이용한 repetition code](https://quantum.cloud.ibm.com/docs/en/tutorials/repetition-codes)

### 실행 중 문제가 생기면

| 증상 | 확인할 내용 |
|:--|:--|
| `ModuleNotFoundError` | 맨 위 설치 셀을 실행하고 커널을 재시작합니다. 설치한 환경과 노트북의 Python 커널이 같은지 확인합니다. |
| `NameError` | 앞의 변수·함수 정의 셀을 실행했는지 확인합니다. **커널 재시작 후 모두 실행**하면 실행 순서를 초기화할 수 있습니다. |
| 회로 그림에서 `pylatexenc` 오류 | `qiskit[visualization]` 설치 여부를 확인합니다. |
| 애니메이션이나 슬라이더가 보이지 않음 | 해당 셀을 다시 실행합니다. 신뢰할 수 있는 이 파일을 열었다면 Jupyter의 Trust 설정도 확인합니다. 슬라이더 대신 함수 인수를 바꾸어 실행할 수 있습니다. |
| 패키지 설치 후에도 예전 오류가 반복됨 | 커널을 재시작하고 맨 위 import 셀의 버전을 확인합니다. |

선택 셀은 해당 소절 안에서 위에서 아래로 실행합니다. 마지막 매개변수 스캔 셀은 독립적으로 실행할 수 있습니다. 모든 코드 셀을 포함해 **커널 재시작 후 모두 실행**하는 흐름으로 구성했습니다.